## Import Libraries

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
import os
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader, WeightedRandomSampler
import torch.optim.lr_scheduler as lr_scheduler
import torchmetrics
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
import numpy as np
import open3d as o3d
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
# from pytorch3d.loss import chamfer_distance
from MTDC_loss import chamfer_distance, adaptive_density_chamfer_distance, torch_normal_consistency_loss, compute_scale_consistency_loss_v2
from MTDC_loss import compute_latent_regularization_v2, compute_local_structure_loss, compute_multiscale_structure_loss_v2, compute_boundary_preservation_loss, point_cloud_uniform_loss, compute_layer_balance_loss
# from MTDC_loss import test
from collections import defaultdict
import json
import time

# Preparation

In [ ]:
# Path handling function
def converted_backslash(original_path):
    converted_path = original_path.replace("\\", "/")
    return converted_path

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Model

In [ ]:
class SkullDataset2(Dataset):
    def __init__(self, data_dir, num_points=5000, is_test=False):
        self.data_dir = data_dir
        self.num_points = num_points
        self.is_test = is_test
        self.files = [f for f in os.listdir(data_dir) if f.endswith(".xyz")]
        
    def __len__(self):
        return len(self.files)
    
    def normalize_pointcloud(self, break_points, fix_points=None):

        break_centroid = np.mean(break_points, axis=0)
        break_points_centered = break_points - break_centroid

        if fix_points is not None and not self.is_test:
            fix_centroid = np.mean(fix_points, axis=0)
            fix_points_centered = fix_points - fix_centroid

            combined_points = np.concatenate([break_points_centered, fix_points_centered], axis=0)
            max_dist = np.max(np.sqrt(np.sum(combined_points**2, axis=1)))

            break_points_norm = break_points_centered / max_dist
            fix_points_norm = fix_points_centered / max_dist

            return break_points_norm, fix_points_norm, break_centroid, fix_centroid, max_dist
        
        else:
            max_dist = np.max(np.sqrt(np.sum(break_points_centered**2, axis=1)))
            break_points_norm = break_points_centered / max_dist

            return break_points_norm, None, break_centroid, None, max_dist


    def __getitem__(self, idx):
        break_file = os.path.join(self.data_dir, self.files[idx])

        break_pcd = o3d.io.read_point_cloud(break_file)

        break_points = np.asarray(break_pcd.points)

        fix_points = None
        if not self.is_test:
            fix_file = os.path.join(self.data_dir, self.files[idx].replace("_break.xyz", "_fix.xyz"))
            fix_pcd = o3d.io.read_point_cloud(fix_file)
            fix_points = np.asarray(fix_pcd.points)
        if len(break_points) > self.num_points:
            idx = np.random.choice(len(break_points), self.num_points, replace=False)
            break_points = break_points[idx]
        if fix_points is not None and len(fix_points) > self.num_points:
            idx = np.random.choice(len(fix_points), self.num_points, replace=False)
            fix_points = fix_points[idx]
            
        break_points, fix_points, break_centroid, fix_centroid, max_dist = self.normalize_pointcloud(break_points, fix_points)
        
        norm_params = {
            'break_centroid': break_centroid, 
            'scale': max_dist,
            'original_break_size': len(break_points)
        }
        if fix_centroid is not None:
            norm_params['fix_centroid'] = fix_centroid
            norm_params['original_fix_size'] = len(fix_points)

        break_points = torch.tensor(break_points, dtype=torch.float32)
        fix_points = torch.tensor(fix_points, dtype=torch.float32) if fix_points is not None else None
        norm_params = {k: torch.tensor(v, dtype=torch.float32) if isinstance(v, (int, float, np.ndarray)) else v 
                        for k, v in norm_params.items()}

        return break_points, fix_points, norm_params
    

def collate_fn(batch):
    break_points, fix_points, norm_params = zip(*batch)
    
    break_points = torch.stack(break_points)
    fix_points = torch.stack(fix_points)
    
    combined_norm_params = {}
    for key in norm_params[0].keys():
        combined_norm_params[key] = torch.stack([p[key] for p in norm_params])
    
    return break_points, fix_points, combined_norm_params


class PointCloudAutoencoder(nn.Module):
    def __init__(self, num_points=5000, latent_dim=128, feature_dim=128):
        super(PointCloudAutoencoder, self).__init__()
        self.num_points = num_points
        self.latent_dim = latent_dim
        self.feature_dim = feature_dim
        
        self.encoder = nn.Sequential(
            ResidualBlock(3, 32), 
            ResidualBlock(32, 64),
            ResidualBlock(64, 128),
            ResidualBlock(128, 256),
            ResidualBlock(256, 512),
            ResidualBlock(512, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(latent_dim),
        )

        self.transformer = nn.Sequential(
            PointTransformerBlock(dim=feature_dim, num_heads=8),
            PointTransformerBlock(dim=feature_dim, num_heads=8)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim * feature_dim, 4096),
            nn.LayerNorm(4096),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(4096, 8192),
            nn.LayerNorm(8192),
            nn.GELU(),
            nn.Dropout(p=0.05),
            nn.Linear(8192, 4096),
            nn.LayerNorm(4096),
            nn.GELU(),
            nn.Linear(4096, num_points * 3),
        )

        self.scale_constraint = nn.Sequential(
            nn.Linear(latent_dim * feature_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )
    
    def encode(self, x):
        x = x.transpose(1, 2)
        x = self.encoder(x)
        x = x.transpose(1, 2)
        for block in self.transformer:
            x = block(x) + x * 0.1
            
        return x.contiguous()
    
    def decode(self, z, apply_scale_constraint=True):
        batch_size = z.size(0) 
        z_flat = z.reshape(batch_size, -1)
        
        x = self.decoder(z_flat)
        x = x.view(batch_size, self.num_points, 3)

        if apply_scale_constraint:
            scale_factor = self.scale_constraint(z_flat) * 0.2 + 1.0
            x = x * scale_factor.unsqueeze(1)
            
        return x
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

    def compute_scale_loss(self, generated_points, target_points):
        """Compute scale consistency loss"""
        gen_scale = torch.sqrt(torch.sum(generated_points**2, dim=-1)).max(dim=-1)[0]
        target_scale = torch.sqrt(torch.sum(target_points**2, dim=-1)).max(dim=-1)[0]
        
        scale_ratio = gen_scale / (target_scale + 1e-8)
        scale_loss = F.mse_loss(scale_ratio, torch.ones_like(scale_ratio))
        
        return scale_loss
    

def square_distance(src, dst):
    """Calculate Euclid distance between each two points."""
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist


def index_points(points, idx):
    """Group points by index"""
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points


def farthest_point_sample(xyz, npoint):
    """Farthest Point Sampling (FPS)"""
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids


def query_ball_point(radius, nsample, xyz, new_xyz):
    """Group points within a radius"""
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long).to(device).view(1, 1, N).repeat([B, S, 1])
    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat([1, 1, nsample])
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx


class PointNetSetAbstraction(nn.Module):
    """PointNet++ Set Abstraction Layer"""
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all=False):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.group_all = group_all
        
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        self.mlp_residuals = nn.ModuleList()
        
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            
            if last_channel == out_channel:
                self.mlp_residuals.append(nn.Identity())
            else:
                self.mlp_residuals.append(nn.Conv2d(last_channel, out_channel, 1))
                
            last_channel = out_channel

    def forward(self, xyz, points):
        xyz = xyz.contiguous()
        if points is not None:
            points = points.contiguous()

        if self.group_all:
            new_xyz = torch.mean(xyz, dim=1, keepdim=True)
            grouped_xyz = xyz.view(xyz.shape[0], 1, xyz.shape[1], 3) - new_xyz.view(xyz.shape[0], 1, 1, 3)
            if points is not None:
                grouped_points = points.view(points.shape[0], 1, points.shape[1], -1).repeat(1, 1, 1, 1)
            else:
                grouped_points = grouped_xyz
        else:
            fps_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, fps_idx)
            
            idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, idx)
            grouped_xyz -= new_xyz.view(xyz.shape[0], self.npoint, 1, 3)
            
            if points is not None:
                grouped_points = index_points(points, idx)
            else:
                grouped_points = grouped_xyz
        
        grouped_points = grouped_points.permute(0, 3, 2, 1)
        
        for i, conv in enumerate(self.mlp_convs):
            identity = grouped_points
            grouped_points = F.relu(self.mlp_bns[i](conv(grouped_points)))
            
            res = self.mlp_residuals[i](identity)
            if res.shape == grouped_points.shape:
                grouped_points = grouped_points + res
        
        new_points = torch.max(grouped_points, 2)[0]
        new_points = new_points.permute(0, 2, 1)
        
        return new_xyz, new_points


class SelfAttention(nn.Module):
    """Self-attention module for point clouds"""
    def __init__(self, dim, num_heads=4):
        super(SelfAttention, self).__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        
        return x


class PointNetPlusPlusEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super(PointNetPlusPlusEncoder, self).__init__()
        
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=3, mlp=[32, 64])
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=64, mlp=[64, 128])
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=128, mlp=[128, feature_dim], group_all=True)
        
        self.attn1 = SelfAttention(64)
        self.attn2 = SelfAttention(128)
        self.attn3 = SelfAttention(feature_dim)
        
        self.fusion = nn.Sequential(
            nn.Linear(64 + 128 + feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, feature_dim)
        )
        
    def forward(self, xyz):
        B, _, N = xyz.shape
        
        xyz1, points1 = self.sa1(xyz, None)
        points1 = self.attn1(points1)
        points1_global = torch.max(points1, dim=1, keepdim=True)[0].expand(-1, N, -1)
        
        xyz2, points2 = self.sa2(xyz1, points1)
        points2 = self.attn2(points2)
        points2_global = torch.max(points2, dim=1, keepdim=True)[0].expand(-1, N, -1)
        
        _, points3 = self.sa3(xyz2, points2)
        points3 = self.attn3(points3)
        points3_global = points3.expand(-1, N, -1)
        
        multi_scale_features = torch.cat([points1_global, points2_global, points3_global], dim=-1)
        fused_features = self.fusion(multi_scale_features)
        
        global_feature = torch.max(fused_features, dim=1)[0]
        
        return global_feature


class PointTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class Diffusion3DUNet(nn.Module):
    """Enhanced 3D UNet with residual connections and attention for diffusion model"""
    def __init__(self, feature_dim=128, time_dim=128, latent_feature_dim=128):
        super(Diffusion3DUNet, self).__init__()
        self.latent_feature_dim = latent_feature_dim
        base_c = latent_feature_dim
        
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim)
        )
        
        self.condition_proj = nn.Sequential(
            nn.Linear(feature_dim, latent_feature_dim),
            nn.ReLU(),
            nn.Linear(latent_feature_dim, latent_feature_dim)
        )
        self.time_proj = nn.Linear(time_dim, latent_feature_dim)
        
        self.scale_predictor = nn.Sequential(
            nn.Linear(feature_dim + time_dim, base_c * 2),
            nn.ReLU(),
            nn.Linear(base_c * 2, base_c),
            nn.ReLU(),
            nn.Linear(base_c, 1),
            nn.Tanh()
        )

        chs = [base_c//1, base_c *2, base_c *4, base_c *8, base_c *8]

        self.enc1 = ResidualBlock(latent_feature_dim, chs[0])
        self.enc2 = ResidualBlock(chs[0], chs[1])
        self.enc3 = ResidualBlock(chs[1], chs[2])
        self.enc4 = ResidualBlock(chs[2], chs[3])
        self.enc5 = ResidualBlock(chs[3], chs[4])

        self.mid_attn1 = SelfAttention(chs[4])
        self.mid_block1 = ResidualBlock(chs[4], chs[4])
        self.mid_attn2 = SelfAttention(chs[4])
        self.mid_ptblock1 = PointTransformerBlock(dim=chs[4], num_heads=4)
        self.mid_block2 = ResidualBlock(chs[4], chs[4])
        self.mid_ptblock2 = PointTransformerBlock(dim=chs[4], num_heads=4)
        
        self.dec5 = ResidualBlock(chs[4] + chs[3], chs[4])
        self.dec4 = ResidualBlock(chs[3] + chs[2], chs[2])
        self.dec3 = ResidualBlock(chs[2] + chs[1], chs[1])
        self.dec2 = ResidualBlock(chs[1] + chs[0], latent_feature_dim)
        self.dec1 = nn.Sequential(
            nn.Conv1d(latent_feature_dim, latent_feature_dim, kernel_size=1),
            nn.Tanh()
        )
        
        self.attn1 = SelfAttention(chs[0])
        self.attn2 = SelfAttention(chs[1])
        self.attn3 = SelfAttention(chs[2])
        self.attn4 = SelfAttention(chs[3])

    def forward(self, z, t, condition):
        """
        z: [B, latent_dim, latent_feature_dim] - latent representation
        t: [B] - Timestep
        condition: [B, feature_dim] - Condition feature from encoder
        """
        B, N, C = z.shape
        
        t_emb = self.time_mlp(t.unsqueeze(-1))
        
        condition_feat = self.condition_proj(condition)
        time_feat = self.time_proj(t_emb)
        
        scale_adjustment = self.scale_predictor(torch.cat([condition, t_emb], dim=-1))

        z = z.transpose(1, 2)
        
        condition_broadcast = (condition_feat + time_feat).unsqueeze(-1).expand(-1, -1, N)
        z = z + condition_broadcast * 0.5

        skip_connections = []
        
        e1 = self.enc1(z)
        e1 = e1 + self.attn1(e1.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e1)
        
        e2 = self.enc2(e1)
        e2 = e2 + self.attn2(e2.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e2)
        
        e3 = self.enc3(e2)
        e3 = e3 + self.attn3(e3.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e3)
        
        e4 = self.enc4(e3)
        e4 = e4 + self.attn4(e4.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e4)
        
        e5 = self.enc5(e4)
        
        mid = e5
        mid = mid + self.mid_attn1(mid.transpose(1, 2)).transpose(1, 2) * 0.1
        mid = self.mid_block1(mid)
        mid = mid + self.mid_attn2(mid.transpose(1, 2)).transpose(1, 2) * 0.1
        mid = self.mid_block2(mid) 
        mid_ptin = mid.transpose(1,2)
        mid_ptout = self.mid_ptblock1(mid_ptin)
        mid_ptout = self.mid_ptblock2(mid_ptout)
        mid = mid_ptout.transpose(1,2)
        
        d5 = self.dec5(torch.cat([mid, skip_connections[3]], dim=1))
        d4 = self.dec4(torch.cat([d5, skip_connections[2]], dim=1))
        d3 = self.dec3(torch.cat([d4, skip_connections[1]], dim=1))
        d2 = self.dec2(torch.cat([d3, skip_connections[0]], dim=1))
        d1 = self.dec1(d2)

        scale_factor = 1.0 + scale_adjustment.unsqueeze(-1) * 0.1
        d1 = d1 * scale_factor

        output = d1.transpose(1, 2)
        
        return output
 

class ResidualBlock(nn.Module):
    """Residual block for 3D UNet"""
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out
    

class EnhancedConditionalDiffusionModel(nn.Module):
    def __init__(self, autoencoder=None, feature_dim=128, beta_schedule='cosine', num_points=5000, num_timesteps=500):
        super(EnhancedConditionalDiffusionModel, self).__init__()
        self.num_points = num_points

        if autoencoder is None:
            raise ValueError("A pretrained autoencoder must be provided")
        self.autoencoder = autoencoder
        self.latent_dim = autoencoder.latent_dim
        self.latent_feature_dim = autoencoder.feature_dim
        self.encoder = PointNetPlusPlusEncoder(feature_dim=feature_dim)
        self.unet = Diffusion3DUNet(feature_dim=feature_dim, time_dim=128, latent_feature_dim=self.latent_feature_dim)
        self.num_timesteps = num_timesteps
        self.latent_dim = autoencoder.latent_dim
        self.latent_feature_dim = autoencoder.feature_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        if beta_schedule == 'linear':
            self.beta = torch.linspace(1e-4, 0.02, self.num_timesteps, device=self.device)
        elif beta_schedule == 'cosine':
            steps = self.num_timesteps + 1
            x = torch.linspace(0, self.num_timesteps, steps, device=self.device)
            alphas_cumprod = torch.cos(((x / self.num_timesteps) + 0.008) * np.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            self.beta = torch.clip(betas, 0.0001, 0.9999)
        
        self.alpha = 1. - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1. - self.alpha_cumprod)

    def forward(self, x, t, condition_points):
        """
        x: The noisy latent representation [B, latent_dim, latent_feature_dim]
        t: Timestep [B]
        condition_points: Condition point cloud [B, N, 3]
        """
        condition = self.encoder(condition_points)
        
        pred_noise = self.unet(x, t / self.num_timesteps, condition)
        
        return pred_noise

    def add_noise(self, z_0, t):
        """Add noise to the latent representation at specified timestep"""
        noise = torch.randn_like(z_0) * 0.8
        sqrt_alpha_cumprod_t = self.sqrt_alpha_cumprod[t].view(-1, 1, 1)
        sqrt_one_minus_alpha_cumprod_t = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1)
        z_t = sqrt_alpha_cumprod_t * z_0 + sqrt_one_minus_alpha_cumprod_t * noise
        
        return z_t, noise

    def sample(self, condition_points, num_points=5000, denormalize_params=None, is_test=False, guidance_scale=1.0):
        self.eval()
        with torch.no_grad():
            if condition_points.dim() == 2:
                condition_points = condition_points.unsqueeze(0)
            elif condition_points.dim() == 4 and condition_points.shape[0] == 1:
                condition_points = condition_points.squeeze(0)
            print("condition_points shape inside model.sample():", condition_points.shape)
            B, N, _ = condition_points.shape
            device = condition_points.device

            z = torch.randn(B, self.latent_dim, self.latent_feature_dim).to(device) * 0.8
            condition = self.encoder(condition_points)

            condition_scale = torch.sqrt(torch.sum(condition_points**2, dim=-1)).max(dim=-1)[0]

            for t in tqdm(reversed(range(self.num_timesteps)), desc="Denoise Progress", total=self.num_timesteps):
                t_tensor = torch.full((B,), t, device=device, dtype=torch.long)
                
                predicted_noise = self.unet(z, t_tensor / self.num_timesteps, condition)

                alpha_t = self.alpha[t]
                alpha_cumprod_t = self.alpha_cumprod[t]
                beta_t = self.beta[t]

                noise = torch.randn_like(z) * 0.5 if t > 0 else torch.zeros_like(z)

                z = (1 / torch.sqrt(alpha_t)) * (
                    z - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise
                ) + torch.sqrt(beta_t) * noise

                if t < 100:
                    current_scale = torch.sqrt(torch.sum(z**2, dim=-1)).max(dim=-1)[0]
                    target_scale = 1.0
                    factor = torch.clamp(target_scale / current_scale, max=1.0)
                    z = z * factor.view(-1, 1, 1)
            
            x = self.autoencoder.decode(z, apply_scale_constraint=True)
            
            if denormalize_params is not None:
                scale = denormalize_params['scale']
                if scale.dim() == 1:
                    scale = scale.view(-1, 1, 1)
                elif scale.dim() == 0:
                    scale = scale.view(1, 1, 1).expand(B, 1, 1)
                centroid_key = 'fix_centroid' if (not is_test and 'fix_centroid' in denormalize_params) else 'break_centroid'
                centroid = denormalize_params[centroid_key]
                if centroid.dim() == 1:
                    centroid = centroid.view(1, 1, 3).expand(B, 1, 3)
                elif centroid.dim() == 2:
                    centroid = centroid.view(B, 1, 3)
                x = x * scale + centroid

                target_scale = condition_scale.view(B, 1, 1) * 0.8
                current_scale = torch.sqrt(torch.sum(x**2, dim=-1)).max(dim=-1)[0].view(B, 1, 1)
                scale_correction = target_scale / (current_scale + 1e-8)
                x = x * scale_correction * scale + centroid

            return x.squeeze(0) if B == 1 else x
        

    def forward_encoder_only(self, latent_or_image):
        """Forward pass using encoder only"""
        return self.forward_encoder_only_impl(latent_or_image)
    

    def forward_encoder_only_impl(self, latent_or_image):
        """
        Retrieve conditional feature embeddings based on input type.
        Supports:
        - Latent space (shape [B, latent_dim, latent_feature_dim])
        - Point cloud (shape [B, N, 3])
        - Extensible to other types (e.g. images, depending on encoder design)
        """
        if latent_or_image.dim() == 3:
            if latent_or_image.shape[1] == self.latent_dim and latent_or_image.shape[2] == self.latent_feature_dim:
                with torch.no_grad():
                    decoded = self.autoencoder.decode(latent_or_image)
                return self.encoder(decoded)
            elif latent_or_image.shape[2] == 3:
                return self.encoder(latent_or_image)
            else:
                raise ValueError(f"forward_encoder_only_impl: Unsupported input shape [B, {latent_or_image.shape[1]}, {latent_or_image.shape[2]}]")
        elif latent_or_image.dim() == 4:
            return self.encoder(latent_or_image)
        else:
            raise ValueError(f"forward_encoder_only_impl: Unsupported input dim={latent_or_image.dim()}, shape={latent_or_image.shape}")

        
    def compute_loss(self, break_points, fix_points, norm_params):
        """Enhanced loss computation with scale regularization"""
        B = break_points.shape[0]
        device = break_points.device
        
        z_0, _ = self.autoencoder(fix_points)
        
        t = torch.randint(0, self.num_timesteps, (B,), device=device)
        
        z_t, noise = self.add_noise(z_0, t)
        
        predicted_noise = self.forward(z_t, t, break_points)
        
        diff_loss = F.mse_loss(predicted_noise, noise)
        recon, _ = self.autoencoder(fix_points)
        ae_loss = F.mse_loss(recon, fix_points)
        
        weight_lambda = 0.2
        total_loss = diff_loss + weight_lambda *ae_loss
        
        if t.mean() < self.num_timesteps * 0.3:
            reconstructed = self.autoencoder.decode(z_t - predicted_noise)
            scale_loss = self.autoencoder.compute_scale_loss(reconstructed, fix_points)
            diff_loss = diff_loss + 0.1 * scale_loss
        
        return diff_loss

In [ ]:
def compute_sample_weights(dataset):
    """
    Computes an importance/difficulty score for each sample and converts it into
    a 1D weight list (weights) for use with WeightedRandomSampler, so that
    important/hard samples are sampled more frequently during training.

    Example weight metrics: degree of point cloud damage (missing ratio),
    Chamfer distance (or other loss values), local density distribution,
    normal vector anomalies, etc.

    Simple example: compute weights based on point cloud density or other features.

    dataset: custom point cloud Dataset that can retrieve individual point cloud data

    return: list or np.array, same length as dataset, values are sampling weights
    """

    weights = []
    for idx in range(len(dataset)):
        sample = dataset[idx]
        break_points = sample[0]
        break_points = break_points.cpu().numpy() if hasattr(break_points, 'cpu') else break_points

        dist_sum = 0
        cnt = 0
        
        for i in range(len(break_points)):
            dists = np.linalg.norm(break_points - break_points[i], axis=1)
            dists = dists[dists > 0]
            if len(dists) > 0:
                dist_sum += np.min(dists)
                cnt += 1
        
        avg_min_dist = dist_sum / cnt if cnt > 0 else 0.001
        weight = avg_min_dist
        weights.append(weight)

    weights = np.array(weights)
    weights = weights / weights.sum()

    return weights

In [ ]:
class EnhancedLatentAlignment(nn.Module):
    """Enhanced latent space alignment mechanism"""
    def __init__(self, autoencoder, alignment_strength=0.1):
        super(EnhancedLatentAlignment, self).__init__()
        self.autoencoder = autoencoder
        self.alignment_strength = alignment_strength
        
        self.latent_dim = autoencoder.latent_dim
        self.feature_dim = autoencoder.feature_dim
        total_dim = self.latent_dim * self.feature_dim
        
        self.latent_adapter = nn.Sequential(
            nn.Linear(total_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, total_dim),
        )
        
        self.distribution_aligner = nn.Sequential(
            nn.Linear(total_dim, total_dim),
            nn.Tanh()
        )
        
        self.mean_estimator = nn.Parameter(torch.zeros(total_dim))
        self.std_estimator = nn.Parameter(torch.ones(total_dim))
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def align_latent_space(self, z):
        """Align latent space distribution"""
        B, N, C = z.shape
        z_original = z.clone()
        
        z_flat = z.reshape(B, -1)
        
        z_adapted = self.latent_adapter(z_flat)
        
        z_aligned = self.distribution_aligner(z_adapted)
        
        z_normalized = self._statistical_alignment(z_aligned)
        
        z_final = z_flat + self.alignment_strength * (z_normalized - z_flat)
        
        return z_final.reshape(B, N, C)
    
    def _statistical_alignment(self, z_flat):
        """Statistical alignment: align latent vectors to the learned distribution"""
        batch_mean = z_flat.mean(dim=0, keepdim=True)
        batch_std = z_flat.std(dim=0, keepdim=True) + 1e-6
        
        z_standardized = (z_flat - batch_mean) / batch_std
        
        z_aligned = z_standardized * self.std_estimator.unsqueeze(0) + self.mean_estimator.unsqueeze(0)
        
        return z_aligned
    
    def update_statistics(self, z_batch):
        """Update distribution statistics (called during training)"""
        with torch.no_grad():
            B, N, C = z_batch.shape
            z_flat = z_batch.reshape(B, -1)
            
            momentum = 0.1
            batch_mean = z_flat.mean(dim=0)
            batch_std = z_flat.std(dim=0)
            
            self.mean_estimator.data = (1 - momentum) * self.mean_estimator.data + momentum * batch_mean
            self.std_estimator.data = (1 - momentum) * self.std_estimator.data + momentum * batch_std

class AlignmentAwareDiffusionModel(EnhancedConditionalDiffusionModel):
    """Diffusion model with integrated alignment mechanism"""
    def __init__(self, autoencoder=None, feature_dim=256, beta_schedule='cosine', 
                 num_points=5000, num_timesteps=500):
        super().__init__(autoencoder, feature_dim, beta_schedule, num_points, num_timesteps)
        
        self.latent_aligner = EnhancedLatentAlignment(autoencoder)
        
    def forward_with_alignment(self, x, t, condition_points, use_alignment=True):
        """Forward pass with alignment"""
        if use_alignment:
            x_aligned = self.latent_aligner.align_latent_space(x)
            return self.forward(x_aligned, t, condition_points)
        else:
            return self.forward(x, t, condition_points)

In [ ]:
def integrate_alignment_to_existing_model(diffusion_model, device):
    """Integrate alignment mechanism into an existing model"""
    
    diffusion_model = diffusion_model.to(device)
    diffusion_model.latent_aligner = EnhancedLatentAlignment(diffusion_model.autoencoder)
    diffusion_model.latent_aligner = diffusion_model.latent_aligner.to(device)
    main_device = next(diffusion_model.parameters()).device
    aligner_device = next(diffusion_model.latent_aligner.parameters()).device
    if main_device != aligner_device:
        print("Warning: device mismatch, correcting...")
        diffusion_model.latent_aligner = diffusion_model.latent_aligner.to(main_device)
        print(f"Aligner moved to: {main_device}")
        
    original_forward = diffusion_model.forward
    
    def forward_with_alignment(x, t, condition_points, use_alignment=True):
        if use_alignment and hasattr(diffusion_model, 'latent_aligner'):
            x_aligned = diffusion_model.latent_aligner.align_latent_space(x)
            return original_forward(x_aligned, t, condition_points)
        else:
            return original_forward(x, t, condition_points)
    
    diffusion_model.forward_with_alignment = forward_with_alignment
    return diffusion_model

In [ ]:
def process_batch_data(data, device):
    """Process batch data and ensure it is on the correct device"""
    try:
        if data is None:
            return None
            
        if isinstance(data, list):
            data = [d for d in data if d is not None]
            if len(data) == 0:
                return None
            data = torch.stack(data)
        
        data = data.to(device)
        
        if len(data.shape) != 3 or data.shape[-1] != 3:
            print(f"Warning: unexpected data shape {data.shape}")
            return None
            
        return data
        
    except Exception as e:
        print(f"Data processing error: {e}")
        return None

## -----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Data Preporcessing

In [ ]:
xyz_training_data = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000")
dataset = SkullDataset2(data_dir = xyz_training_data, num_points = 5000)
weights = compute_sample_weights(dataset) 

dataset_indices = list(range(len(dataset)))
train_indices, val_indices = train_test_split(dataset_indices, test_size=0.2, random_state=42)

train_weights = [weights[i] for i in train_indices]
val_weights = [weights[i] for i in val_indices]

train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)
val_sampler = WeightedRandomSampler(
    weights=val_weights,
    num_samples=len(val_weights),
    replacement=True
)

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 64,
    sampler = train_sampler,
    shuffle = False, 
    collate_fn = collate_fn,
    # pin_memory=True, # 加速CPU -> GPU數據傳輸
    num_workers=0
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 64,
    sampler = val_sampler,
    shuffle = False, 
    collate_fn = collate_fn,
    num_workers=0
)
print("len(dataset):", len(dataset))
print("max(train_indices):", max(train_indices))
print("max(val_indices):", max(val_indices))
print("min(train_indices):", min(train_indices))
print("min(val_indices):", min(val_indices))
print(f"train_dataloader length: {len(train_dataloader)}")
print(f"val_dataloader length: {len(val_dataloader)}")

Data Augmentation

In [ ]:
def augment(batch_points):
    theta = np.random.uniform(0, 2*np.pi)
    rot = torch.tensor([[np.cos(theta), -np.sin(theta), 0],
                        [np.sin(theta),  np.cos(theta), 0],[0, 0, 1]], dtype=batch_points.dtype, device=batch_points.device)
    return torch.matmul(batch_points, rot)

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Define Training AE Progress

In [ ]:
class AdaptiveGradientClipper:
    def __init__(self, model, max_norm=1.0, percentile=95):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        
    def clip_gradients(self, loss):
        """Adaptive gradient clipping"""
        loss.backward()
        
        total_norm = 0
        param_count = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        total_norm = total_norm ** (1. / 2)
        
        self.grad_history.append(total_norm)
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 1.2)
        else:
            clip_norm = self.max_norm
        
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class StabilizedGradientClipper:
    """Stricter gradient clipping"""
    def __init__(self, model, max_norm=0.8, percentile=90):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        self.stability_counter = 0
        
    def clip_gradients_enhanced(self):
        total_norm = 0
        param_count = 0
        
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        
        if param_count == 0:
            return 0.0, self.max_norm
            
        total_norm = total_norm ** 0.5
        self.grad_history.append(total_norm)
        
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 0.9)
        else:
            clip_norm = self.max_norm
        
        if total_norm > clip_norm * 2.0:
            self.stability_counter += 1
        else:
            self.stability_counter = max(0, self.stability_counter - 1)
        
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class EnhancedTrainingLoop:
    def __init__(self, model, optimizer, scheduler=None):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.grad_clipper = StabilizedGradientClipper(model, max_norm=1.0)
        self.loss_history = []
        
    def training_step(self, break_points, fix_points, norm_params):
        """Enhanced training step"""
        self.optimizer.zero_grad()
        
        loss = self.compute_enhanced_loss(break_points, fix_points, norm_params)
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"Warning: Invalid loss detected: {loss}")
            return None
        
        grad_norm, clip_norm = self.grad_clipper.clip_gradients(loss)
        
        if grad_norm > 5.0:
            print(f"Warning: Large gradient norm: {grad_norm:.4f}")
            self.optimizer.zero_grad()
            for param_group in self.optimizer.param_groups:
                param_group['lr'] *= 0.95
            return None
        
        self.optimizer.step()
        if self.scheduler:
            self.scheduler.step()
        
        self.loss_history.append(loss.item())
        if len(self.loss_history) > 1000:
            self.loss_history.pop(0)
        
        return {
            'loss': loss.item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            'lr': self.optimizer.param_groups['lr']
        }
    

    def progressive_timestep_sampling(self, batch_size, device):
        """Progressive timestep sampling"""
        progress = min(len(self.loss_history) / 10000, 1.0)
        
        if progress < 0.3:
            min_t = int(self.model.num_timesteps * 0.5)
        elif progress < 0.7:
            min_t = 0
        else:
            min_t = 0
            weights = torch.exp(-torch.arange(self.model.num_timesteps) / 100.0)
            t = torch.multinomial(weights, batch_size, replacement=True)
            return t.to(device)
        
        return torch.randint(min_t, self.model.num_timesteps, (batch_size,), device=device)
    
    def compute_time_weights(self, t):
        """Compute timestep weights"""
        weights = 1.0 / (t.float() + 1e-8)
        weights = weights / weights.max()
        return weights
    
    def balance_losses(self, diff_loss, recon_loss, scale_loss):
        """Dynamic loss balancing"""
        progress = min(len(self.loss_history) / 5000, 1.0)
        
        diff_weight = 0.6 + 0.3 * progress
        
        recon_weight = 0.4 - 0.2 * progress
        
        scale_weight = 0.1
        
        total_loss = (diff_weight * diff_loss + 
                     recon_weight * recon_loss + 
                     scale_weight * scale_loss)
        
        return total_loss

Warmup + Restarts lr adapt

In [ ]:
class CosineWarmRestarts:
    def __init__(self, optimizer, T_0=1000, T_mult=2, eta_min=1e-6, base_lr=2e-5):
        self.optimizer = optimizer
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.base_lr = base_lr
        self.T_cur = 0
        self.T_i = T_0
        self.cycle = 0
        
    def step(self):
        self.T_cur += 1
        
        if self.T_cur >= self.T_i:
            self.cycle += 1
            self.T_cur = 0
            self.T_i = self.T_i * self.T_mult
            
        lr = self.eta_min + (self.base_lr - self.eta_min) * \
             (1 + np.cos(np.pi * self.T_cur / self.T_i)) / 2
        
        if self.T_cur == 0 and self.cycle > 0:
            lr = self.base_lr * (0.8 ** self.cycle)  
            
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        return lr

In [ ]:
def get_linear_warmup_scheduler(optimizer, warmup_epochs):
    def lr_lambda(epoch):
        return min((epoch+1)/warmup_epochs, 1.0)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

Early Stop

In [ ]:
class EarlyStoppingWithStability:
    """Early stopping with stability monitoring"""
    def __init__(self, patience=7, min_delta=0.001, stability_threshold=5):
        self.patience = patience
        self.min_delta = min_delta
        self.stability_threshold = stability_threshold
        self.counter = 0
        self.best_loss = float('inf')
        self.unstable_epochs = 0
        
    def __call__(self, val_loss, grad_norm):
        improved = val_loss < self.best_loss - self.min_delta
        
        if improved:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        
        if grad_norm > 2.0:
            self.unstable_epochs += 1
        else:
            self.unstable_epochs = max(0, self.unstable_epochs - 1)
        
        patience_exceeded = self.counter >= self.patience
        too_unstable = self.unstable_epochs >= self.stability_threshold
        
        return patience_exceeded or too_unstable

Loss Function

In [ ]:
def compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder):
    """Compute enhanced multi-term losses"""
    losses = {}
    
    try:
        losses['dcd_loss'], _ = adaptive_density_chamfer_distance(
            recon_points, fix_points, k=8, density_method='knn', adaptive_weight=True
        )

        losses['cd_loss'], _ = chamfer_distance(recon_points, fix_points)

        losses['mse_loss'] = F.mse_loss(recon_points, fix_points)
        
        losses['structure_loss'] = compute_multiscale_structure_loss_v2(recon_points, fix_points)

        losses['boundary_loss'] = compute_boundary_preservation_loss(recon_points, fix_points)

    except Exception as e:
        print(f"Loss computation error: {e}")
        device = recon_points.device
        losses = {
            'dcd_loss': torch.tensor(0.1, device=device, requires_grad=True),
            'structure_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'boundary_loss': torch.tensor(0.01, device=device, requires_grad=True),
        }
    
    return losses

def compute_enhanced_losses_amp_compatible(recon_points, fix_points, latent_z, autoencoder):
    """AMP-compatible loss computation"""
    losses = {}
    
    try:
        with torch.cuda.amp.autocast(enabled=False):
            recon_fp32 = recon_points.float()
            fix_fp32 = fix_points.float()
            
            losses['dcd_loss'], _ = adaptive_density_chamfer_distance(
                recon_fp32, fix_fp32, k=8, density_method='knn', adaptive_weight=True
            )
            
            losses['cd_loss'], _ = chamfer_distance(recon_fp32, fix_fp32)
            
            losses['normal_loss'] = torch_normal_consistency_loss(recon_fp32, fix_fp32, k=8)
        
        losses['mse_loss'] = F.mse_loss(recon_points, fix_points)
        losses['huber_loss'] = F.smooth_l1_loss(recon_points, fix_points)
        
        losses['scale_loss'] = compute_scale_consistency_loss_v2(recon_points, fix_points)
        
        losses['latent_reg'] = compute_latent_regularization_v2(latent_z)
        
        with torch.cuda.amp.autocast(enabled=False):
            losses['local_structure_loss'] = compute_local_structure_loss(recon_fp32, fix_fp32)
        
        for key, loss in losses.items():
            if not loss.requires_grad:
                print(f"Warning: {key} has no gradient information")
                losses[key] = loss.clone().requires_grad_(True)
        
    except Exception as e:
        print(f"Loss computation error: {e}")
        device = recon_points.device
        losses = {
            'dcd_loss': torch.tensor(1.0, device=device, requires_grad=True),
            'cd_loss': torch.tensor(0.1, device=device, requires_grad=True),
            'mse_loss': F.mse_loss(recon_points, fix_points),
            'huber_loss': F.smooth_l1_loss(recon_points, fix_points),
            'scale_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'normal_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'latent_reg': compute_latent_regularization_v2(latent_z),
            'local_structure_loss': torch.tensor(0.01, device=device, requires_grad=True)
        }
    
    return losses

def compute_weighted_total_loss(losses, current_epoch, total_epochs):
    """Dynamic weighted loss computation"""
    progress = min(current_epoch / total_epochs, 1.0)
    
    weights = {
        'dcd_loss': 0.78,
        'structure_loss': 0.22,
    }
    
    total_loss = sum(weights[key] * loss for key, loss in losses.items() 
                    if key in weights and torch.is_tensor(loss))
    
    return total_loss

In [ ]:
def train_enhanced_autoencoder(autoencoder, train_dataloader, val_dataloader, 
                              num_epochs=35, lr=2e-5, device='cuda',
                              save_path_ae="autoencoder_pretrained__v8_0824_0035.pth",
                              log_dir=converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0824_0035\logs\autoencoder_v8_0824_0142")):
    """
    Enhanced AutoEncoder pretraining function.
    Integrates all improvements: stabilized training, multi-loss functions, full monitoring.
    """
    
    autoencoder.to(device)
    
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(os.path.dirname(save_path_ae), exist_ok=True)
    
    save_dir = os.path.dirname(save_path_ae)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    else:
        default_save_dir = os.path.join(log_dir, 'models')
        os.makedirs(default_save_dir, exist_ok=True)
        save_path_ae = os.path.join(default_save_dir, save_path_ae)
        print(f"Save path adjusted to: {save_path_ae}")
    
    writer = SummaryWriter(log_dir=log_dir, flush_secs=30)
    
    optimizer = torch.optim.AdamW(
        autoencoder.parameters(), 
        lr=1e-4, 
        betas=(0.9, 0.999),
        weight_decay=1e-4,
        eps=1e-8
    )

    warmup_epochs = 15
    scheduler = StableCosineLR(optimizer, T_max=num_epochs, eta_min=lr*0.01)
    warmup_sched = get_linear_warmup_scheduler(optimizer, warmup_epochs=warmup_epochs)
    main_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs-warmup_epochs, eta_min=1e-6
    )

    grad_clipper = StabilizedGradientClipper(autoencoder, max_norm=2.0)

    scaler = GradScaler() if device == 'cuda' else None
    scaler = None
    monitor = EnhancedTrainingMonitor(log_dir)

    metrics_history = {
        'train': defaultdict(list),
        'val': defaultdict(list),
        'lr': [],
        'grad_norm': [],
        'best_metrics': {'epoch': 0, 'loss': float('inf')}
    }
    
    print("Starting Enhanced AutoEncoder pretraining...")
    print(f"Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}")
    print(f"Device: {device}, LR: {lr}, Epochs: {num_epochs}")
    print("-" * 40)
    
    for epoch in range(num_epochs):
        
        autoencoder.train()
        train_metrics = train_epoch_enhanced(
            autoencoder, train_dataloader, optimizer, 
            grad_clipper, device, epoch, num_epochs, scaler
        )
        
        autoencoder.eval()
        val_metrics = validate_epoch_enhanced(
            autoencoder, val_dataloader, device, epoch
        )
        
        if epoch < warmup_epochs:
            warmup_sched.step()
            current_lr = optimizer.param_groups[0]['lr']
        else:
            main_sched.step()
            current_lr = optimizer.param_groups[0]['lr']
        
        for key, value in train_metrics.items():
            metrics_history['train'][key].append(value)
        for key, value in val_metrics.items():
            metrics_history['val'][key].append(value)
        
        metrics_history['lr'].append(current_lr)
        if 'grad_norm' in train_metrics:
            metrics_history['grad_norm'].append(train_metrics['grad_norm'])
        
        log_to_tensorboard(writer, train_metrics, val_metrics, current_lr, epoch)
        
        print_epoch_summary(epoch, num_epochs, train_metrics, val_metrics, current_lr)
        
        monitor.check_training_stability(train_metrics, val_metrics, epoch)
        
        current_loss = val_metrics.get('total_loss', train_metrics.get('total_loss', float('inf')))
        if current_loss < metrics_history['best_metrics']['loss']:
            metrics_history['best_metrics']['loss'] = current_loss
            metrics_history['best_metrics']['epoch'] = epoch
            
            best_model_path = save_path_ae.replace('.pth', '_best.pth')
            save_checkpoint(autoencoder, optimizer, scheduler, metrics_history, 
                          best_model_path, epoch, is_best=True)
            print(f"New best model saved: {best_model_path}")

        if (epoch + 1) % 30 == 0 or epoch == num_epochs - 1:
            checkpoint_path = save_path_ae.replace('.pth', f'_epoch_{epoch+1}.pth')
            torch.save(autoencoder.state_dict(), checkpoint_path)
        
        diagnose_training(train_metrics, val_metrics, epoch, metrics_history)
        
        if epoch > warmup_epochs:
            adjust_training_strategy(optimizer, scheduler, metrics_history, epoch)

        grad_clipper.grad_history.clear()
        if device == "cuda":
            torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("AutoEncoder pretraining complete!")
    
    final_save_checkpoint(autoencoder, optimizer, scheduler, metrics_history, save_path_ae)
    
    generate_training_report(metrics_history, log_dir, num_epochs)
    
    plot_training_curves(metrics_history, log_dir)
    
    writer.close()
    
    print(f"Best model (Epoch {metrics_history['best_metrics']['epoch']}): "
          f"Loss = {metrics_history['best_metrics']['loss']:.6f}")
    print(f"Model saved to: {save_path_ae}")
    print(f"Logs saved to: {log_dir}")
    
    return extract_metrics_for_return(metrics_history)


def train_epoch_enhanced(autoencoder, train_dataloader, optimizer, grad_clipper, 
                        device, epoch, total_epochs, scaler=None):
    """Enhanced training epoch function"""
    
    epoch_metrics = defaultdict(list)
    
    progress_bar = tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{total_epochs}")
    
    for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
        
        if fix_points is None:
            continue
        
        fix_points = process_batch_data(fix_points, device)
        if fix_points is None:
            continue
            
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                recon_points, latent_z = autoencoder(fix_points)
                losses = compute_enhanced_losses_amp_compatible(recon_points, fix_points, latent_z, autoencoder)
                total_loss = compute_weighted_total_loss(losses, epoch, total_epochs)
        
            if not torch.isfinite(total_loss):
                print(f"Warning: Invalid loss detected (Epoch {epoch}, Batch {batch_idx})")
                continue
        
            scaler.scale(total_loss).backward()
            
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), max_norm = 2.0)
            clip_norm = 2.0

            if grad_norm > 50.0:
                print(f"Warning: Gradient explosion (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            scaler.step(optimizer)
            scaler.update()
        
        else:
            recon_points, latent_z = autoencoder(fix_points)
            losses = compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder)
            total_loss = compute_weighted_total_loss(losses, epoch, total_epochs)
            
            if not torch.isfinite(total_loss):
                print(f"Warning: Invalid loss detected (Epoch {epoch}, Batch {batch_idx})")
                continue
            
            total_loss.backward()
            grad_norm, clip_norm = grad_clipper.clip_gradients_enhanced()
            
            if grad_norm > 50.0:
                print(f"Warning: Gradient explosion (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            optimizer.step()

        batch_metrics = {
            'total_loss': total_loss.item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
        }
        
        for key, value in batch_metrics.items():
            epoch_metrics[key].append(value)
        
        if batch_idx % 10 == 0:
            progress_bar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'DCD': f"{losses['dcd_loss'].item():.4f}",
                'GradNorm': f"{grad_norm:.2f}"
            })
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

def validate_epoch_enhanced(autoencoder, val_dataloader, device, epoch):
    """Enhanced validation epoch function"""
    
    epoch_metrics = defaultdict(list)
    
    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"Validation Epoch {epoch+1}")
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            fix_points = process_batch_data(fix_points, device)
            if fix_points is None:
                continue
            
            recon_points, latent_z = autoencoder(fix_points)
            
            losses = compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder)
            total_loss = compute_weighted_total_loss(losses, epoch, 50)
            
            batch_metrics = {
                'total_loss': total_loss.item(),
                **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
            }
            
            for key, value in batch_metrics.items():
                epoch_metrics[key].append(value)
            
            if batch_idx % 10 == 0:
                progress_bar.set_postfix({
                    'Val_Loss': f"{total_loss.item():.4f}",
                    'Val_DCD': f"{losses['dcd_loss'].item():.4f}"
                })
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}


class StableCosineLR:
    """Stable cosine annealing scheduler"""
    def __init__(self, optimizer, T_max=50, eta_min=1e-6, warmup_epochs=5):
        self.optimizer = optimizer
        self.T_max = T_max
        self.eta_min = eta_min
        self.warmup_epochs = warmup_epochs
        self.base_lr = optimizer.param_groups[0]['lr']
        self.current_epoch = 0
        
    def step(self):
        if self.current_epoch < self.warmup_epochs:
            lr = self.base_lr * (self.current_epoch + 1) / self.warmup_epochs
        else:
            progress = (self.current_epoch - self.warmup_epochs) / (self.T_max - self.warmup_epochs)
            lr = self.eta_min + (self.base_lr - self.eta_min) * \
                 (1 + np.cos(np.pi * progress)) / 2
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        self.current_epoch += 1
        return lr


class EnhancedTrainingMonitor:
    """Enhanced training monitor"""
    def __init__(self, log_dir):
        self.log_dir = log_dir
        self.stability_window = 50
        
    def check_training_stability(self, train_metrics, val_metrics, epoch):
        """Check training stability"""
        warnings = []
        
        if train_metrics.get('total_loss', 0) > 10.0:
            warnings.append("Training loss too large")
        
        if train_metrics.get('grad_norm', 0) > 10.0:
            warnings.append("Gradient norm too large")
        
        train_loss = train_metrics.get('total_loss', 0)
        val_loss = val_metrics.get('total_loss', 0)
        if val_loss > train_loss * 2:
            warnings.append("Possible overfitting")
        
        if warnings:
            print(f"Epoch {epoch+1} Warning: {', '.join(warnings)}")


def log_to_tensorboard(writer, train_metrics, val_metrics, lr, epoch):
    """Log to TensorBoard"""
    
    for key, value in train_metrics.items():
        writer.add_scalar(f'Train/{key}', value, epoch)
    
    for key, value in val_metrics.items():
        writer.add_scalar(f'Val/{key}', value, epoch)
    
    writer.add_scalar('Learning_Rate', lr, epoch)
    
    if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
        writer.add_scalars('Loss_Comparison', {
            'Train': train_metrics['total_loss'],
            'Val': val_metrics['total_loss']
        }, epoch)

def print_epoch_summary(epoch, total_epochs, train_metrics, val_metrics, lr):
    """Print epoch summary"""
    print(f"\nEpoch {epoch+1}/{total_epochs} Summary:")
    print(f"Learning Rate: {lr:.2e}")
    
    print(f"Train - Total Loss: {train_metrics.get('total_loss', 0):.4f}, "
          f"DCD: {train_metrics.get('dcd_loss', 0):.4f}, "
          f"Grad Norm: {train_metrics.get('grad_norm', 0):.2f}")
    
    print(f"Val - Total Loss: {val_metrics.get('total_loss', 0):.4f}, "
          f"DCD: {val_metrics.get('dcd_loss', 0):.4f}")
    
    print(f"Reference - Train CD: {train_metrics.get('cd_loss', 0):.4f}, "
          f"Val CD: {val_metrics.get('cd_loss', 0):.4f}")

def save_checkpoint(model, optimizer, scheduler, metrics_history, 
                   save_path, epoch, is_best=False):
    """Save checkpoint"""
    checkpoint = {
        'model_state_dict': model.state_dict(),
    }
    
    torch.save(checkpoint, save_path)

def final_save_checkpoint(model, optimizer, scheduler, metrics_history, save_path):
    """Final save"""
    save_checkpoint(model, optimizer, scheduler, metrics_history, save_path, 
                   metrics_history['best_metrics']['epoch'], is_best=False)

def diagnose_training(train_metrics, val_metrics, epoch, metrics_history):
    """Training diagnostics"""
    if epoch < 5:
        return
    
    recent_train_losses = metrics_history['train']['total_loss'][-5:]
    if len(recent_train_losses) >= 5:
        if all(recent_train_losses[i] >= recent_train_losses[i-1] for i in range(1, 5)):
            print("Warning: Training loss rising consecutively, consider adjusting learning rate")

def adjust_training_strategy(optimizer, scheduler, metrics_history, epoch):
    """Dynamically adjust training strategy"""
    if epoch > 20:
        recent_losses = metrics_history['train']['total_loss'][-10:]
        if len(recent_losses) >= 10:
            loss_variance = np.var(recent_losses)
            if loss_variance < 1e-6:
                current_lr = optimizer.param_groups[0]['lr']
                new_lr = current_lr * 0.8
                for param_group in optimizer.param_groups:
                    param_group['lr'] = new_lr
                print(f"Learning rate adjusted: {current_lr:.2e} -> {new_lr:.2e}")

def generate_training_report(metrics_history, log_dir, num_epochs):
    """Generate training report"""
    report = {
        'training_summary': {
            'total_epochs': num_epochs,
            'best_epoch': metrics_history['best_metrics']['epoch'],
            'best_loss': metrics_history['best_metrics']['loss'],
        },
        'final_metrics': {
            'train': {k: v[-1] if v else 0 for k, v in metrics_history['train'].items()},
            'val': {k: v[-1] if v else 0 for k, v in metrics_history['val'].items()}
        },
        'training_stability': {
            'loss_variance': np.var(metrics_history['train']['total_loss'][-20:]) if len(metrics_history['train']['total_loss']) >= 20 else 0,
            'avg_grad_norm': np.mean(metrics_history['grad_norm'][-20:]) if len(metrics_history['grad_norm']) >= 20 else 0
        }
    }
    
    with open(f'{log_dir}/training_report.json', 'w') as f:
        json.dump(report, f, indent=2)

def plot_training_curves(metrics_history, log_dir):
    """Plot training curves"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    axes[0,0].plot(metrics_history['train']['total_loss'], label='Train', color='blue')
    axes[0,0].plot(metrics_history['val']['total_loss'], label='Val', color='red')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    axes[0,1].plot(metrics_history['train']['dcd_loss'], label='Train DCD', color='green')
    axes[0,1].plot(metrics_history['val']['dcd_loss'], label='Val DCD', color='orange')
    axes[0,1].set_title('DCD Loss')
    axes[0,1].legend()
    axes[0,1].grid(True)
    
    axes[0,2].plot(metrics_history['train']['cd_loss'], label='Train CD', color='purple')
    axes[0,2].plot(metrics_history['val']['cd_loss'], label='Val CD', color='brown')
    axes[0,2].set_title('Chamfer Distance')
    axes[0,2].legend()
    axes[0,2].grid(True)
    
    axes[1,0].plot(metrics_history['grad_norm'], color='red', alpha=0.7)
    axes[1,0].set_title('Gradient Norm')
    axes[1,0].grid(True)
    
    axes[1,1].plot(metrics_history['lr'], color='black')
    axes[1,1].set_title('Learning Rate')
    axes[1,1].grid(True)
    
    if 'mse_loss' in metrics_history['train']:
        axes[1,2].plot(metrics_history['train']['mse_loss'], label='Train MSE', color='cyan')
        axes[1,2].plot(metrics_history['val']['mse_loss'], label='Val MSE', color='magenta')
        axes[1,2].set_title('MSE Loss')
        axes[1,2].legend()
        axes[1,2].grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{log_dir}/training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()

def extract_metrics_for_return(metrics_history):
    """Extract metrics for return"""
    return (
        metrics_history['train']['cd_loss'],
        metrics_history['train']['mse_loss'],
        metrics_history['train']['dcd_loss'],
        metrics_history['val']['cd_loss'],
        metrics_history['val']['mse_loss'],
        metrics_history['val']['dcd_loss']
    )

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Training AE

In [ ]:
torch.cuda.empty_cache()
# train_model(EnhancedConditionalDiffusionModel, dataloader)

In [ ]:
if __name__ == "__main__":
    # os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

    autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=128,
        feature_dim=128
    )
    
    base_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107")
    models_dir = os.path.join(base_dir, "models")
    os.makedirs(models_dir, exist_ok=True)
    save_path = os.path.join(models_dir, "autoencoder_pretrained_v8_0912_2137.pth")
    log_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\logs\autoencoder_v8_0912_2137")
    
    train_results = train_enhanced_autoencoder(
        autoencoder=autoencoder,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        num_epochs=60,
        lr=2e-4,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        save_path_ae=save_path,
        log_dir=log_dir
    )
    
    print("AutoEncoder trained!")

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Define Training Diffusion Model Progress

In [ ]:
class DiffusionGradientClipper:
    """Gradient clipper optimized for diffusion models"""
    def __init__(self, model, max_norm=1.0, percentile=95):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        
    def clip_gradients(self, loss):
        """Gradient clipping dedicated to diffusion models"""
        loss.backward()
        
        total_norm = 0
        param_count = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        total_norm = total_norm ** (1. / 2)
        
        self.grad_history.append(total_norm)
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 1.2)
        else:
            clip_norm = self.max_norm
        
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class StabilizedDiffusionGradientClipper:
    """More stable gradient clipper for diffusion models"""
    def __init__(self, model, max_norm=0.8, percentile=90):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        self.stability_counter = 0
        
    def clip_gradients_enhanced(self):
        total_norm = 0
        param_count = 0
        
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        
        if param_count == 0:
            return 0.0, self.max_norm
            
        total_norm = total_norm ** 0.5
        self.grad_history.append(total_norm)
        
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 0.9)
        else:
            clip_norm = self.max_norm
        
        if total_norm > clip_norm * 2.0:
            self.stability_counter += 1
        else:
            self.stability_counter = max(0, self.stability_counter - 1)
        
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class EnhancedDiffusionTrainingLoop:
    """Diffusion model training loop"""
    def __init__(self, diffusion_model, optimizer, scheduler=None, num_timesteps=1000):
        self.diffusion_model = diffusion_model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.grad_clipper = StabilizedDiffusionGradientClipper(diffusion_model, max_norm=1.0)
        self.loss_history = []
        self.num_timesteps = num_timesteps
        
    def training_step(self, break_points, fix_points, norm_params):
        """Diffusion model training step - core change: predicting noise instead of reconstruction"""
        self.optimizer.zero_grad()
        
        loss_dict = self.compute_diffusion_loss(break_points, fix_points, norm_params)
        total_loss = loss_dict['total_loss']
        
        if torch.isnan(total_loss) or torch.isinf(total_loss):
            print(f"Warning: Invalid diffusion loss detected: {total_loss}")
            return None
        
        grad_norm, clip_norm = self.grad_clipper.clip_gradients(total_loss)
        
        if grad_norm > 3.0:
            print(f"Warning: Large gradient norm in diffusion training: {grad_norm:.4f}")
            self.optimizer.zero_grad()
            for param_group in self.optimizer.param_groups:
                param_group['lr'] *= 0.9
            return None
        
        self.optimizer.step()
        if self.scheduler:
            self.scheduler.step()
        
        self.loss_history.append(total_loss.item())
        if len(self.loss_history) > 1000:
            self.loss_history.pop(0)
        
        return {
            'total_loss': total_loss.item(),
            'noise_loss': loss_dict['noise_loss'].item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            'lr': self.optimizer.param_groups[0]['lr'],
            **{k: v.item() if torch.is_tensor(v) else v 
               for k, v in loss_dict.items() if k != 'total_loss'}
        }
    
    def compute_diffusion_loss(self, break_points, fix_points, norm_params):
        """Compute diffusion model dedicated loss function"""
        B = fix_points.shape[0]
        device = fix_points.device
        
        with torch.no_grad():
            z_0 = self.diffusion_model.autoencoder.encode(fix_points)
        
        t = self.progressive_timestep_sampling(B, device)
        
        z_t, noise= self.diffusion_model.add_noise(z_0, t)
        
        predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
        
        losses = self.compute_diffusion_losses(predicted_noise, noise, z_t, z_0, t, fix_points)
        
        total_loss = self.balance_diffusion_losses(losses, t)
        
        return {
            'total_loss': total_loss,
            'noise_loss': losses['noise_loss'],
            **losses
        }
    
    def progressive_timestep_sampling(self, batch_size, device):
        """Progressive timestep sampling strategy"""
        progress = min(len(self.loss_history) / 5000, 1.0)
        
        if progress < 0.2:
            min_t = int(self.num_timesteps * 0.7)
            max_t = self.num_timesteps
        elif progress < 0.6:
            min_t = 0
            max_t = self.num_timesteps
        else:
            weights = torch.exp(-torch.arange(self.num_timesteps, dtype=torch.float) / 200.0)
            t = torch.multinomial(weights, batch_size, replacement=True)
            return t.to(device)
        
        return torch.randint(min_t, max_t, (batch_size,), device=device)
    
    def compute_diffusion_losses(self, predicted_noise, true_noise, z_t, z_0, t, x_0):
        """Compute various losses for the diffusion model"""
        losses = {}
        
        losses['noise_loss'] = F.mse_loss(predicted_noise, true_noise)
        
        time_weights = self.compute_time_weights(t)
        weighted_noise_loss = F.mse_loss(predicted_noise, true_noise, reduction='none')
        losses['weighted_noise_loss'] = (weighted_noise_loss * time_weights.view(-1, 1, 1)).mean()
        
        losses['huber_noise_loss'] = F.smooth_l1_loss(predicted_noise, true_noise)
        
        if hasattr(self.diffusion_model, 'predict_x0') and self.diffusion_model.predict_x0:
            predicted_x0 = self.diffusion_model.predict_original_from_noise(z_t, predicted_noise, t)
            losses['x0_loss'] = F.mse_loss(predicted_x0, z_0)
        
        if hasattr(self.diffusion_model, 'v_parameterization'):
            v_target = self.compute_v_target(z_0, true_noise, t)
            predicted_v = self.diffusion_model.predict_v(z_t, t)
            losses['v_loss'] = F.mse_loss(predicted_v, v_target)
        
        losses['latent_reg'] = torch.mean(z_t ** 2) * 1e-4
        
        return losses
    
    def compute_time_weights(self, t):
        """Compute timestep weights to balance learning across different noise levels"""
        alpha_t = self.diffusion_model.scheduler.alphas_cumprod[t]
        snr = alpha_t / (1 - alpha_t)
        
        weights = 1.0 / (snr + 1e-8)
        weights = weights / weights.max()
        return weights
    
    def compute_v_target(self, x0, noise, t):
        """Compute target for v-parameterization"""
        alpha_t = self.diffusion_model.scheduler.alphas_cumprod[t].view(-1, 1, 1)
        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_alpha_t = torch.sqrt(1 - alpha_t)
        
        v = sqrt_alpha_t * noise - sqrt_one_minus_alpha_t * x0
        return v
    
    def balance_diffusion_losses(self, losses, t):
        """Dynamically balance diffusion losses"""
        progress = min(len(self.loss_history) / 3000, 1.0)
        
        weights = {
            'noise_loss': 0.7,
            'weighted_noise_loss': 0.2,
            'huber_noise_loss': 0.05,
            'latent_reg': 0.02,
        }
        
        if 'x0_loss' in losses:
            weights['x0_loss'] = 0.1 * progress
            weights['noise_loss'] = 0.6
        
        if 'v_loss' in losses:
            weights['v_loss'] = 0.15 * progress
            weights['noise_loss'] = 0.55
        
        total_loss = sum(weights.get(key, 0) * loss 
                        for key, loss in losses.items() 
                        if torch.is_tensor(loss))
        
        return total_loss

class CosineWarmupScheduler:
    """Cosine annealing scheduler for diffusion models"""
    def __init__(self, optimizer, warmup_steps=1000, max_steps=10000, base_lr=1e-4, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_step = 0
        
    def step(self):
        """Execute learning rate scheduling"""
        self.current_step += 1
        
        if self.current_step <= self.warmup_steps:
            lr = self.base_lr * self.current_step / self.warmup_steps
        else:
            progress = (self.current_step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            progress = min(progress, 1.0)
            lr = self.min_lr + (self.base_lr - self.min_lr) * \
                 (1 + np.cos(np.pi * progress)) / 2
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        return lr

class DiffusionEarlyStopping:
    """Early stopping mechanism for diffusion models"""
    def __init__(self, patience=15, min_delta=0.001, stability_threshold=10):
        self.patience = patience
        self.min_delta = min_delta
        self.stability_threshold = stability_threshold
        self.counter = 0
        self.best_loss = float('inf')
        self.unstable_epochs = 0
        
    def __call__(self, val_loss, grad_norm):
        improved = val_loss < self.best_loss - self.min_delta
        
        if improved:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        
        if grad_norm > 1.5:
            self.unstable_epochs += 1
        else:
            self.unstable_epochs = max(0, self.unstable_epochs - 1)
        
        patience_exceeded = self.counter >= self.patience
        too_unstable = self.unstable_epochs >= self.stability_threshold
        
        return patience_exceeded or too_unstable

def compute_diffusion_losses_enhanced(predicted_noise, true_noise, z_t, z_0, t, 
                                    x_0, diffusion_model, timestep_weights=None):
    """Enhanced diffusion loss computation function"""
    losses = {}
    
    try:
        losses['noise_mse'] = F.mse_loss(predicted_noise, true_noise)
        
        losses['noise_huber'] = F.smooth_l1_loss(predicted_noise, true_noise)
        
        if timestep_weights is not None:
            weighted_loss = F.mse_loss(predicted_noise, true_noise, reduction='none')
            losses['weighted_noise'] = (weighted_loss * timestep_weights.view(-1, 1, 1)).mean()
        
        losses['noise_l1'] = F.l1_loss(predicted_noise, true_noise)
        
        losses['latent_consistency'] = torch.mean(torch.abs(z_t - z_0))
        
        if predicted_noise.requires_grad:
            grad_norm = torch.norm(torch.autograd.grad(
                predicted_noise.sum(), predicted_noise, 
                create_graph=True, retain_graph=True)[0])
            losses['grad_penalty'] = grad_norm * 1e-4

        losses['latent_reg'] = torch.mean(torch.abs(z_t)) * 1e-4

        
    except Exception as e:
        print(f"Diffusion loss computation error: {e}")
        device = predicted_noise.device
        losses = {
            'noise_mse': F.mse_loss(predicted_noise, true_noise),
            'noise_huber': F.smooth_l1_loss(predicted_noise, true_noise),
            'latent_consistency': torch.tensor(0.01, device=device, requires_grad=True),
            'latent_reg': torch.tensor(0.1, device=device, requires_grad=True),
        }
    
    return losses

def compute_weighted_diffusion_loss(losses, current_step, total_steps):
    """Dynamic weighted diffusion loss computation"""
    progress = min(current_step / total_steps, 1.0)
    
    weights = {
        'noise_mse': 0.78,
        'weighted_noise': 0.10,
        'latent_consistency': 0.05,
        'noise_huber': 0.04,
        'grad_penalty': 0.02,
        'latent_reg': 0.01,
    }
    
    if progress > 0.5:
        weights['noise_huber'] *= 1.2
        weights['weighted_noise'] *= 1.3
    
    total_loss = sum(weights.get(key, 0) * loss 
                    for key, loss in losses.items() 
                    if key in weights and torch.is_tensor(loss))
    
    return total_loss

In [ ]:
class SimpleGradientClipper:
    """Simplified gradient clipper to avoid complex memory operations"""
    def __init__(self, model, max_norm=1.0):
        self.model = model
        self.max_norm = max_norm
    
    def clip_gradients_simple(self):
        """Simplified gradient clipping"""
        try:
            total_norm = torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), self.max_norm
            )
            return total_norm, total_norm
        except Exception as e:
            print(f"Gradient clipping error: {e}")
            return 0.0, 0.0

In [ ]:
def compute_distribution_regularization(z):
    """Compute distribution regularization loss"""
    try:
        z_flat = z.reshape(z.shape[0], -1)
        
        mean = z_flat.mean(dim=1, keepdim=True)
        std = z_flat.std(dim=1, keepdim=True) + 1e-6
        
        kl_loss = 0.5 * (mean.pow(2) + std.pow(2) - 2 * torch.log(std) - 1).mean()
        
        return kl_loss
    except:
        return torch.tensor(0.0, device=z.device, requires_grad=True)

def compute_alignment_consistency_loss(z_original, z_aligned, x_original, autoencoder):
    """Compute alignment consistency loss"""
    try:
        with torch.no_grad():
            recon_original = autoencoder.decode(z_original)
        
        recon_aligned = autoencoder.decode(z_aligned)
        
        consistency_loss = F.mse_loss(recon_aligned, recon_original)
        
        return consistency_loss
    except:
        return torch.tensor(0.0, device=z_aligned.device, requires_grad=True)

In [ ]:
def train_enhanced_diffusion_model(diffusion_model, train_dataloader, val_dataloader,
                                 num_epochs=100, lr=1e-4, device='cuda',
                                 save_path_diffusion="diffusion_model_v1.pth",
                                 log_dir="./logs/diffusion_training"):
    """
    Enhanced diffusion model training function.
    Core change: focused on noise prediction rather than data reconstruction.
    """
    
    diffusion_model.to(device)
    
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(os.path.dirname(save_path_diffusion), exist_ok=True)
    
    writer = SummaryWriter(log_dir=log_dir, flush_secs=30)
    
    optimizer = torch.optim.AdamW(
        diffusion_model.parameters(), 
        lr=lr, 
        betas=(0.9, 0.999),
        weight_decay=1e-4,
        eps=1e-8
    )
    
    total_steps = len(train_dataloader) * num_epochs
    scheduler = CosineWarmupScheduler(
        optimizer, 
        warmup_steps=total_steps // 10,
        max_steps=total_steps,
        base_lr=lr,
        min_lr=lr * 0.01
    )
    
    early_stopping = DiffusionEarlyStopping(patience=50, min_delta=0.0005)
    
    grad_clipper = StabilizedDiffusionGradientClipper(diffusion_model, max_norm=1.5)
    
    scaler = None
    
    metrics_history = {
        'train': defaultdict(list),
        'val': defaultdict(list),
        'lr': [],
        'grad_norm': [],
        'best_metrics': {'epoch': 0, 'loss': float('inf')}
    }
    
    print("Starting diffusion model training...")
    print(f"Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}")
    print(f"Device: {device}, LR: {lr}, Epochs: {num_epochs}")
    print(f"Diffusion timesteps: {diffusion_model.num_timesteps}")
    print("-" * 50)
    
    global_step = 0
    
    for epoch in range(num_epochs):
        
        diffusion_model.train()
        train_metrics = train_diffusion_epoch(
            diffusion_model, train_dataloader, optimizer, scheduler,
            grad_clipper, device, epoch, num_epochs, scaler, global_step
        )
        
        global_step += len(train_dataloader)
        
        diffusion_model.eval()
        val_metrics = validate_diffusion_epoch(
            diffusion_model, val_dataloader, device, epoch
        )
        
        current_lr = scheduler.optimizer.param_groups[0]['lr']
        
        for key, value in train_metrics.items():
            metrics_history['train'][key].append(value)
        for key, value in val_metrics.items():
            metrics_history['val'][key].append(value)
        
        metrics_history['lr'].append(current_lr)
        if 'grad_norm' in train_metrics:
            metrics_history['grad_norm'].append(train_metrics['grad_norm'])
        
        log_diffusion_metrics(writer, train_metrics, val_metrics, current_lr, epoch)
        
        print_diffusion_summary(epoch, num_epochs, train_metrics, val_metrics, current_lr)
        
        current_loss = val_metrics.get('total_loss', train_metrics.get('total_loss', float('inf')))
        if current_loss < metrics_history['best_metrics']['loss']:
            metrics_history['best_metrics']['loss'] = current_loss
            metrics_history['best_metrics']['epoch'] = epoch
            
            best_model_path = save_path_diffusion.replace('.pth', '_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': diffusion_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.__dict__,
                'metrics_history': metrics_history,
                'loss': current_loss
            }, best_model_path)
            print(f"New best diffusion model saved: {best_model_path}")
        
        if (epoch + 1) % 100 == 0 or epoch == num_epochs - 1:
            checkpoint_path = save_path_diffusion.replace('.pth', f'_epoch_{epoch+1}.pth')
            torch.save(diffusion_model.state_dict(), checkpoint_path)
        
        if early_stopping(val_metrics.get('total_loss', float('inf')), 
                         train_metrics.get('grad_norm', 0)):
            print(f"Diffusion model early stopping at Epoch {epoch+1}")
            break
        
        if device == "cuda":
            torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("Diffusion model training complete!")
    
    torch.save({
        'model_state_dict': diffusion_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics_history': metrics_history,
    }, save_path_diffusion)
    
    generate_diffusion_training_report(metrics_history, log_dir, num_epochs)
    
    plot_diffusion_training_curves(metrics_history, log_dir)
    
    writer.close()
    
    print(f"Best model (Epoch {metrics_history['best_metrics']['epoch']}): "
          f"Loss = {metrics_history['best_metrics']['loss']:.6f}")
    print(f"Diffusion model saved to: {save_path_diffusion}")
    print(f"Training logs saved to: {log_dir}")
    
    return extract_diffusion_metrics_for_return(metrics_history)


def train_diffusion_epoch(diffusion_model, train_dataloader, optimizer, scheduler,
                         grad_clipper, device, epoch, total_epochs, scaler=None, global_step=0):
    """Diffusion model training epoch"""
    
    epoch_metrics = defaultdict(list)
    
    progress_bar = tqdm(train_dataloader, desc=f"Diffusion Training Epoch {epoch+1}/{total_epochs}")
    
    step = 0
    for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
        
        if fix_points is None:
            continue
        
        try:
            fix_points = process_batch_data(fix_points, device)
            if fix_points is None:
                continue
                
            break_points = process_batch_data(break_points, device)
            if break_points is None:
                continue
            
            fix_points = fix_points.to(device)
            break_points = break_points.to(device)
            
        except Exception as e:
            print(f"Data device transfer error: {e}")
            continue
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                with torch.no_grad():
                    z_0 = diffusion_model.autoencoder.encode(fix_points)
                
                B = z_0.shape[0]
                device = z_0.device
                
                t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
                
                z_t, noise = diffusion_model.add_noise(z_0, t)
                
                predicted_noise = diffusion_model.forward(z_t, t, break_points)
                
                losses = compute_diffusion_losses_enhanced(
                    predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
                )
                total_loss = compute_weighted_diffusion_loss(losses, global_step + step, 
                                                           len(train_dataloader) * total_epochs)
            
            if not torch.isfinite(total_loss):
                print(f"Warning: Invalid diffusion loss detected (Epoch {epoch}, Batch {batch_idx})")
                continue
            
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), max_norm=1.5)
            
            if grad_norm > 10.0:
                print(f"Warning: Diffusion model gradient explosion (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            scaler.step(optimizer)
            scaler.update()
        
        else:
            with torch.no_grad():
                z_0 = diffusion_model.autoencoder.encode(fix_points)
            
            B = z_0.shape[0]
            t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
            
            z_t, noise = diffusion_model.add_noise(z_0, t)
            
            predicted_noise = diffusion_model.forward(z_t, t, break_points)
            
            losses = compute_diffusion_losses_enhanced(
                predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
            )
            total_loss = compute_weighted_diffusion_loss(losses, global_step + step,
                                                       len(train_dataloader) * total_epochs)
            
            if not torch.isfinite(total_loss):
                continue
            
            total_loss.backward()
            grad_norm, clip_norm = grad_clipper.clip_gradients_enhanced()
            
            if grad_norm > 10.0:
                optimizer.zero_grad()
                continue
            
            optimizer.step()
        
        scheduler.step()
        
        batch_metrics = {
            'total_loss': total_loss.item(),
            'noise_loss': losses.get('noise_mse', total_loss).item(),
            'grad_norm': grad_norm,
            **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
        }
        
        for key, value in batch_metrics.items():
            epoch_metrics[key].append(value)
        
        if batch_idx % 10 == 0:
            progress_bar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                'GradNorm': f"{grad_norm:.2f}"
            })
        
        step += 1
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

def validate_diffusion_epoch(diffusion_model, val_dataloader, device, epoch):
    """Diffusion model validation epoch"""
    
    epoch_metrics = defaultdict(list)
    
    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"Diffusion Validation Epoch {epoch+1}")
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            try:
                fix_points = process_batch_data(fix_points, device)
                if fix_points is None:
                    continue
                    
                break_points = process_batch_data(break_points, device)
                if break_points is None:
                    continue
                
                fix_points = fix_points.to(device)
                break_points = break_points.to(device)
                
            except Exception as e:
                print(f"Validation data device transfer error: {e}")
                continue
            
            z_0 = diffusion_model.autoencoder.encode(fix_points)
            
            B = z_0.shape[0]
            t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
            
            z_t, noise= diffusion_model.add_noise(z_0, t)
            
            predicted_noise = diffusion_model.forward(z_t, t, break_points)
            
            losses = compute_diffusion_losses_enhanced(
                predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
            )
            total_loss = compute_weighted_diffusion_loss(losses, 0, 1)
            
            batch_metrics = {
                'total_loss': total_loss.item(),
                'noise_loss': losses.get('noise_mse', total_loss).item(),
                **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
            }
            
            for key, value in batch_metrics.items():
                epoch_metrics[key].append(value)
            
            if batch_idx % 10 == 0:
                progress_bar.set_postfix({
                    'Val_Loss': f"{total_loss.item():.4f}",
                    'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                })
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}


def log_diffusion_metrics(writer, train_metrics, val_metrics, lr, epoch):
    """Log diffusion model training metrics to TensorBoard"""
    
    for key, value in train_metrics.items():
        writer.add_scalar(f'Diffusion_Train/{key}', value, epoch)
    
    for key, value in val_metrics.items():
        writer.add_scalar(f'Diffusion_Val/{key}', value, epoch)
    
    writer.add_scalar('Diffusion_Learning_Rate', lr, epoch)
    
    if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
        writer.add_scalars('Diffusion_Loss_Comparison', {
            'Train': train_metrics['total_loss'],
            'Val': val_metrics['total_loss']
        }, epoch)

def print_diffusion_summary(epoch, total_epochs, train_metrics, val_metrics, lr):
    """Print diffusion model training summary"""
    print(f"\nDiffusion Model Epoch {epoch+1}/{total_epochs} Summary:")
    print(f"Learning Rate: {lr:.2e}")
    
    print(f"Train - Total Loss: {train_metrics.get('total_loss', 0):.4f}, "
          f"Noise Loss: {train_metrics.get('noise_loss', 0):.4f}, "
          f"Grad Norm: {train_metrics.get('grad_norm', 0):.2f}")
    
    print(f"Val - Total Loss: {val_metrics.get('total_loss', 0):.4f}, "
          f"Noise Loss: {val_metrics.get('noise_loss', 0):.4f}")

def generate_diffusion_training_report(metrics_history, log_dir, num_epochs):
    """Generate diffusion model training report"""
    report = {
        'training_summary': {
            'model_type': 'Diffusion Model',
            'total_epochs': num_epochs,
            'best_epoch': metrics_history['best_metrics']['epoch'],
            'best_loss': metrics_history['best_metrics']['loss'],
        },
        'final_metrics': {
            'train': {k: v[-1] if v else 0 for k, v in metrics_history['train'].items()},
            'val': {k: v[-1] if v else 0 for k, v in metrics_history['val'].items()}
        },
        'training_stability': {
            'loss_variance': np.var(metrics_history['train']['total_loss'][-20:]) if len(metrics_history['train']['total_loss']) >= 20 else 0,
            'avg_grad_norm': np.mean(metrics_history['grad_norm'][-20:]) if len(metrics_history['grad_norm']) >= 20 else 0
        }
    }
    
    with open(f'{log_dir}/diffusion_training_report.json', 'w') as f:
        json.dump(report, f, indent=2)

def plot_diffusion_training_curves(metrics_history, log_dir):
    """Plot diffusion model training curves"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    axes[0,0].plot(metrics_history['train']['total_loss'], label='Train Total', color='blue')
    axes[0,0].plot(metrics_history['val']['total_loss'], label='Val Total', color='red')
    axes[0,0].set_title('Total Loss (Diffusion)')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    if 'noise_loss' in metrics_history['train']:
        axes[0,1].plot(metrics_history['train']['noise_loss'], label='Train Noise', color='green')
        axes[0,1].plot(metrics_history['val']['noise_loss'], label='Val Noise', color='orange')
        axes[0,1].set_title('Noise Prediction Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)
    
    if 'noise_mse' in metrics_history['train']:
        axes[0,2].plot(metrics_history['train']['noise_mse'], label='Train MSE', color='purple')
        axes[0,2].plot(metrics_history['val']['noise_mse'], label='Val MSE', color='brown')
        axes[0,2].set_title('MSE Noise Loss')
        axes[0,2].legend()
        axes[0,2].grid(True)
    
    axes[1,0].plot(metrics_history['grad_norm'], color='red', alpha=0.7)
    axes[1,0].set_title('Gradient Norm')
    axes[1,0].grid(True)
    
    axes[1,1].plot(metrics_history['lr'], color='black')
    axes[1,1].set_title('Learning Rate')
    axes[1,1].grid(True)
    
    if 'noise_huber' in metrics_history['train']:
        axes[1,2].plot(metrics_history['train']['noise_huber'], label='Train Huber', color='cyan')
        axes[1,2].plot(metrics_history['val']['noise_huber'], label='Val Huber', color='magenta')
        axes[1,2].set_title('Huber Noise Loss')
        axes[1,2].legend()
        axes[1,2].grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{log_dir}/diffusion_training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()

def extract_diffusion_metrics_for_return(metrics_history):
    """Extract diffusion model training metrics for return"""
    return (
        metrics_history['train']['total_loss'],
        metrics_history['train'].get('noise_loss', []),
        metrics_history['val']['total_loss'],
        metrics_history['val'].get('noise_loss', []),
    )

In [ ]:
class IntegratedMultiStageTrainingManager:
    """Integrated multi-stage training manager based on existing optimization pipeline"""
    def __init__(self, diffusion_model, train_dataloader, val_dataloader, device, log_dir):
        self.diffusion_model = diffusion_model
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = device
        self.log_dir = log_dir
        
        self.stage_log_dirs = {
            'stage1': os.path.join(log_dir, 'stage1_alignment'),
            'stage2': os.path.join(log_dir, 'stage2_joint'), 
            'stage3': os.path.join(log_dir, 'stage3_diffusion')
        }
        
        for log_dir_path in self.stage_log_dirs.values():
            os.makedirs(log_dir_path, exist_ok=True)
        
        self.writers = {}
        
        self.optimizers = {}
        self.schedulers = {}
        self.early_stoppers = {}
        self.grad_clippers = {}
        
        self.stage_metrics = {
            'stage1': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []},
            'stage2': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []},
            'stage3': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []}
        }
        
    def setup_stage_components(self):
        """Set up training components for each stage"""
        
        clip_values = {'stage1': 1.0, 'stage2': 1.5, 'stage3': 0.8}

        stage1_params = list(self.diffusion_model.latent_aligner.parameters())
        self.optimizers['stage1'] = torch.optim.AdamW(
            stage1_params, lr=2e-5, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        stage2_params = (list(self.diffusion_model.latent_aligner.parameters()) + 
                        list(self.diffusion_model.unet.parameters()) +
                        list(self.diffusion_model.encoder.parameters()))
        self.optimizers['stage2'] = torch.optim.AdamW(
            stage2_params, lr=1.5e-5, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        stage3_params = (list(self.diffusion_model.unet.parameters()) +
                        list(self.diffusion_model.encoder.parameters()))
        self.optimizers['stage3'] = torch.optim.AdamW(
            stage3_params, lr=8e-6, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        for stage in ['stage1', 'stage2', 'stage3']:
            epochs = {'stage1': 20, 'stage2': 80, 'stage3': 100}[stage]
            total_steps = len(self.train_dataloader) * epochs
            
            if stage == 'stage1':
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 5,
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.1
                )
            elif stage == 'stage2':
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 10,
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.01
                )
            else:
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 20,
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.05
                )
            
            patience = {'stage1': 15, 'stage2': 30, 'stage3': 25}[stage]
            min_delta = {'stage1': 0.005, 'stage2': 0.0002, 'stage3': 0.0001}[stage]
            self.early_stoppers[stage] = DiffusionEarlyStopping(
                patience=patience, min_delta=min_delta
            )

            self.grad_clippers[stage] = SimpleGradientClipper(
                self.diffusion_model, max_norm=clip_values[stage]
            )
            
            self.writers[stage] = SummaryWriter(
                log_dir=self.stage_log_dirs[stage], flush_secs=30
            )
    

    def _save_stage_checkpoint(self, stage, epoch, loss, is_final=False):
        """Save stage checkpoint"""
        try:
            best_path = os.path.join(self.stage_log_dirs[stage], f'{stage}_best.pth')
            final_path = os.path.join(self.stage_log_dirs[stage], f'{stage}_best.pth')
            checkpoint_data = {
                'model_state_dict': self.diffusion_model.state_dict(),
            }
            if is_final:
                torch.save(checkpoint_data, final_path)
                print(f"{stage.upper()} Final model saved: Epoch {epoch+1}, Loss: {loss:.4f}")
                print(f"Checkpoint path: {final_path}")
            else:
                torch.save(checkpoint_data, best_path)
                print(f"{stage.upper()} Best model saved: Epoch {epoch+1}, Loss: {loss:.4f}")
                print(f"Checkpoint path: {best_path}")
            
        except Exception as e:
            print(f"Checkpoint save error: {e}")


    def _freeze_params_for_stage1(self):
        """Freeze all parameters except latent_aligner"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = True

    def _freeze_params_for_stage2(self):
        """Freeze AE; allow latent_aligner, unet, and encoder to train"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = True

    def _freeze_params_for_stage3(self):
        """Freeze all parameters except unet and encoder"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = True

    def dynamic_loss_weights(self, epoch, total_epochs):
        progress = epoch / total_epochs
        alignment_weight = 0.80 - progress * 0.15
        structure_weight = 0.10 + progress + 0.10
        dist_reg_weight = 0.08  + progress + 0.05
        condition_weight = 0.07                     
        return alignment_weight, structure_weight, dist_reg_weight, condition_weight


    def run_integrated_multi_stage_training(self, stage1_epochs=20, stage2_epochs=80, stage3_epochs=100):
        """Execute integrated multi-stage training"""
        
        print("=== Starting Integrated Multi-Stage Diffusion Model Training ===")
        self.setup_stage_components()
        
        print(f"\n--- Stage 1: Latent Space Alignment ({stage1_epochs} epochs) ---")
        self._train_integrated_stage1(stage1_epochs)
        
        for writer in self.writers.values():
            writer.close()
        
        self._generate_multi_stage_report()
        
        print("\n=== Integrated Multi-Stage Training Complete ===")
        return self.stage_metrics


    def check_gpu_health(self):
        """Check GPU health status"""
        if not torch.cuda.is_available():
            return False
        
        try:
            test_tensor = torch.randn(10, 10, device=self.device)
            result = test_tensor @ test_tensor.T
            
            if torch.isnan(result).any():
                return False
            
            del test_tensor, result
            torch.cuda.empty_cache()
            
            return True
            
        except Exception as e:
            print(f"GPU health check failed: {e}")
            return False


    def _train_integrated_stage1(self, num_epochs):
        """Stage 1: Integrated alignment training"""
        
        if not self.check_gpu_health():
            print("GPU status abnormal, cannot start training")
            return

        self._freeze_params_for_stage1()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            self.diffusion_model.train()
            train_metrics = self._train_stage1_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            self.diffusion_model.eval()
            val_metrics = self._validate_stage1_epoch(epoch, total_epochs=20)
            
            self._log_stage_metrics('stage1', epoch, train_metrics, val_metrics)
            
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage1', epoch, current_loss, is_final=False)
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage1', epoch, current_loss, is_final=True)
            
            if self.early_stoppers['stage1'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"Stage 1 early stopping at Epoch {epoch+1}")
                break
            
            if self.device == "cuda":
                torch.cuda.empty_cache()

    def _train_stage1_epoch(self, epoch, total_epochs, global_step):
        """Stage 1 single epoch training"""
        
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-1: Epoch {epoch+1}/{total_epochs}")
        
        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            
            if fix_points is None:
                continue
            
            if consecutive_errors > 3:  
                print(f"Continous error: ({consecutive_errors})，interrupt epoch{epoch+1}")
                break
            if critical_error_count > 1:
                print(f"Critical error: ({critical_error_count})，interrupt epoch{epoch+1}")
                break            

            try:
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                
                if fix_points is None or break_points is None:
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"Invalid fix_points，skipping batch{batch_idx}")
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"Invalid break_points，skipping batch{batch_idx}")
                    consecutive_errors += 1
                    continue
                
                self.optimizers['stage1'].zero_grad()
            
                with torch.no_grad():
                    try:
                        z_original = self.diffusion_model.autoencoder.encode(fix_points)
                        condition_embedding = self.diffusion_model.encoder(break_points)
                        if torch.isnan(z_original).any():
                            print(f"AE enocder generates NaN，skipping batch{batch_idx}")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"AE encodes error: {e}")
                        consecutive_errors += 1
                        continue

                try:
                    z_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_original)
                    if torch.isnan(z_aligned).any():
                        print(f"z_aligned contain NaN，skipping batch{batch_idx}")
                        consecutive_errors += 1
                        continue
                except Exception as e:
                    print(f"Aligning Error: {e}")
                    consecutive_errors += 1
                    continue

                try:
                    recon_aligned = self.diffusion_model.autoencoder.decode(z_aligned)
                    if torch.isnan(recon_aligned).any():
                        print(f"recon_aligned contain NaN，skipping batch{batch_idx}")
                        consecutive_errors += 1
                        continue

                    alignment_weight, structure_weight, dist_reg_weight, condition_weight = self.dynamic_loss_weights(epoch, total_epochs)

                    alignment_loss = F.mse_loss(recon_aligned, fix_points) * alignment_weight
                    structure_loss = F.mse_loss(z_aligned, z_original.detach()) * structure_weight
                    dist_reg_loss = compute_distribution_regularization(z_aligned) * dist_reg_weight
                    condition_consistency = F.mse_loss(
                        self.diffusion_model.forward_encoder_only(z_aligned),
                        condition_embedding.detach()
                    ) * condition_weight

                    total_loss = alignment_loss + structure_loss + dist_reg_loss + condition_consistency


                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"Ivalid loss, skip batch{batch_idx}")
                        consecutive_errors += 1
                        continue
                    
                except Exception as e:
                    print(f"Computing loss error: {e}")
                    consecutive_errors += 1
                    continue
            
                
                try:
                    total_loss.backward()
                    
                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(self.diffusion_model.parameters(), max_grad_norm)
                    
                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"total_loss is NaN/Inf，drop batch{batch_idx}")
                        self.optimizers['stage1'].zero_grad()
                        continue

                    all_grad_ok = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            all_grad_ok = False
                            break

                    if not all_grad_ok:
                        print(f"Found NaN/Inf-gradient，Drop batch{batch_idx}")
                        self.optimizers['stage1'].zero_grad()
                        continue
                            
                    self.optimizers['stage1'].step()
                    self.schedulers['stage1'].step()
                    
                    self.diffusion_model.latent_aligner.update_statistics(z_original.detach())
                    
                    batch_metrics = {
                        'alignment_loss': self.safe_tensor_to_float(alignment_loss),
                        'structure_loss': self.safe_tensor_to_float(structure_loss),
                        'dist_reg_loss': self.safe_tensor_to_float(dist_reg_loss),
                        'condition_consistency': self.safe_tensor_to_float(condition_consistency),
                        'total_loss': self.safe_tensor_to_float(total_loss),
                        'grad_norm': self.safe_tensor_to_float(grad_norm)
                    }
                    for key, value in batch_metrics.items():
                        if torch.is_tensor(value):
                            value = float(value.detach().cpu().numpy())
                        epoch_metrics[key].append(value)
                    
                    consecutive_errors = 0
                    critical_error_count = 0

                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Align_Loss': f"{alignment_loss.item():.4f}",
                            'Struct_Loss': f"{structure_loss.item():.4f}",
                            'Reg_Loss': f"{dist_reg_loss.item():.4f}",
                            'Condition_Loss': f"{condition_consistency.item():.4f}",
                            'Total_Loss': f"{grad_norm:.2f}",
                            'GradNorm': f"{grad_norm:.2f}",
                        })

                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"CUDA error in batch{batch_idx}: {str(e)[:100]}")
                        consecutive_errors += 1
                        
                        self.optimizers['stage1'].zero_grad()
                        
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        
                        time.sleep(0.5)
                        continue
                    else:
                        raise e
                
                step += 1
                
            except Exception as e:
                print(f"Training error batch {batch_idx}: {e}")
                critical_error_count += 1
                consecutive_errors += 1
                if critical_error_count > 1:
                    print("Too much critical error, interrupt training immediately!")
                    break

                self.optimizers['stage1'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue
        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print("Warning: No successly trained batch")
            return {
                'total_loss': float('nan'), 
                'alignment_loss': float('nan'),
                'structure_loss': float('nan'),
                'grad_norm': float('nan')
            }
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    def _validate_stage1_epoch(self, epoch, total_epochs):
        """Stage 1 validation"""
        
        epoch_metrics = defaultdict(list)
        error_count = 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage3-1: Val Epoch {epoch+1}")
            
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                
                if error_count > 2:
                    print("Too much valid error, valid early stop on validating")
                    break

                try:
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)
                    if fix_points is None:
                        error_count += 1
                        continue
                    
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)
                    
                    z_original = self.diffusion_model.autoencoder.encode(fix_points)
                    condition_embedding = self.diffusion_model.encoder(break_points)
                    z_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_original)
                    recon_aligned = self.diffusion_model.autoencoder.decode(z_aligned)
                    
                    alignment_weight, structure_weight, dist_reg_weight, condition_weight = self.dynamic_loss_weights(epoch, total_epochs)

                    alignment_loss = F.mse_loss(recon_aligned, fix_points) * alignment_weight
                    structure_loss = F.mse_loss(z_aligned, z_original.detach()) * structure_weight
                    dist_reg_loss = compute_distribution_regularization(z_aligned) * dist_reg_weight
                    condition_consistency = F.mse_loss(
                        self.diffusion_model.forward_encoder_only(z_aligned),
                        condition_embedding.detach()
                    ) * condition_weight
                    
                    total_loss = alignment_loss + structure_loss + dist_reg_loss + condition_consistency
                    
                    batch_metrics = {
                        'alignment_loss': self.safe_tensor_to_float(alignment_loss),
                        'structure_loss': self.safe_tensor_to_float(structure_loss),
                        'dist_reg_loss': self.safe_tensor_to_float(dist_reg_loss),
                        'condition_consistency': self.safe_tensor_to_float(condition_consistency),
                        'total_loss': self.safe_tensor_to_float(total_loss)
                    }
                    
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)
                    
                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Val_Align': f"{alignment_loss.item():.4f}",
                            'Val_Struct': f"{structure_loss.item():.4f}",
                            'Val_DistReg': f"{dist_reg_loss.item():.4f}",
                            'Val_CondCons': f"{condition_consistency.item():.4f}"
                        })
                    
                except Exception as e:
                    print(f"Stage3-1 Error valid batch{batch_idx}: {e}")
                    if self.device == "cuda":
                        torch.cuda.empty_cache()
                    continue
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    def _train_integrated_stage2(self, num_epochs):
        """Stage 2: Integrated joint training"""
        
        self._freeze_params_for_stage2()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            self.diffusion_model.train()
            train_metrics = self._train_stage2_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            self.diffusion_model.eval()
            val_metrics = self._validate_stage2_epoch(epoch)
            
            self._log_stage_metrics('stage2', epoch, train_metrics, val_metrics)
            
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage2', epoch, current_loss, is_final=False)
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage2', epoch, current_loss, is_final=True)
            
            if self.early_stoppers['stage2'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"Stage 2 early stopping at Epoch {epoch+1}")
                break
            
            if self.device == "cuda":
                torch.cuda.empty_cache()

    def _train_stage2_epoch(self, epoch, total_epochs, global_step):
        """Stage 2 improved epoch training: full error checking and professional logging."""
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-2: Epoch {epoch+1}/{total_epochs}")

        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue

            if consecutive_errors > 3:
                print(f"[Epoch {epoch} Batch {batch_idx}] Consecutive error limit reached. Stopping epoch early.")
                break
            if critical_error_count > 1:
                print(f"[Epoch {epoch} Batch {batch_idx}] Too many critical errors. Stopping epoch early.")
                break

            try:
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                if fix_points is None or break_points is None:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                    consecutive_errors += 1
                    continue
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                fix_points = fix_points.to(self.device)
                break_points = break_points.to(self.device)

                self.optimizers['stage2'].zero_grad()

                with torch.no_grad():
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        if torch.isnan(z_0).any():
                            print(f"[Epoch {epoch} Batch {batch_idx}] Latent encoding produced NaN, skipping batch.")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Error during latent encoding: {e}")
                        consecutive_errors += 1
                        continue

                try:
                    z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                    if torch.isnan(z_0_aligned).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Aligned latent contains NaN, skipping batch.")
                        consecutive_errors += 1
                        continue

                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    alignment_consistency = compute_alignment_consistency_loss(
                        z_0, z_0_aligned, fix_points, self.diffusion_model.autoencoder
                    ) * 0.1

                    total_loss = compute_weighted_diffusion_loss(
                        losses, global_step + step, len(self.train_dataloader) * total_epochs
                    )
                    total_loss = total_loss + alignment_consistency
                    
                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is not finite, skipping batch.")
                        consecutive_errors += 1
                        continue
                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Error during forward/loss process: {e}")
                    consecutive_errors += 1
                    continue

                try:
                    total_loss.backward()

                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        self.diffusion_model.parameters(), max_grad_norm
                    )

                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is NaN/Inf after backward, dropping batch.")
                        self.optimizers['stage2'].zero_grad()
                        consecutive_errors += 1
                        continue

                    grad_valid = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            grad_valid = False
                            break
                    if not grad_valid:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Invalid gradient detected, dropping batch.")
                        self.optimizers['stage2'].zero_grad()
                        consecutive_errors += 1
                        continue

                    self.optimizers['stage2'].step()
                    self.schedulers['stage2'].step()
               
                    batch_metrics = {
                        'total_loss': total_loss.item(),
                        'noise_loss': losses.get('noise_mse', total_loss).item(),
                        'alignment_consistency': alignment_consistency.item(),
                        'grad_norm': grad_norm,
                        **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
                    }
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)

                    consecutive_errors = 0
                    critical_error_count = 0

                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Loss': f"{total_loss.item():.4f}",
                            'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                            'GradNorm': f"{grad_norm:.2f}"
                        })
                    step += 1

                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"[Epoch {epoch} Batch {batch_idx}] CUDA runtime error: {str(e)[:100]}")
                        consecutive_errors += 1
                        self.optimizers['stage2'].zero_grad()
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        time.sleep(0.5)
                        continue
                    else:
                        raise e

                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Unexpected training error: {e}")
                    consecutive_errors += 1
                    critical_error_count += 1
                    if critical_error_count > 1:
                        print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                        break
                    self.optimizers['stage2'].zero_grad()
                    if self.device == "cuda":
                        try:
                            torch.cuda.empty_cache()
                        except:
                            pass
                    continue
                
            except Exception as e:
                print(f"[Epoch {epoch} Batch {batch_idx}] Critical batch processing error: {e}")
                critical_error_count += 1
                consecutive_errors += 1
                if critical_error_count > 1:
                    print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                    break
                self.optimizers['stage2'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue

        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print(f"[Epoch {epoch}] WARNING: No successfully trained batches this epoch.")
            return {
                'total_loss': float('nan'), 
                'noise_loss': float('nan'),
                'alignment_consistency': float('nan'),
                'grad_norm': float('nan')
            }

        return {key: np.mean(values) for key, values in epoch_metrics.items()}


    def _validate_stage2_epoch(self, epoch):
        """Stage 2 validation epoch (robust, professional log output)."""
        epoch_metrics = defaultdict(list)
        self.diffusion_model.eval()

        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage2-Val-Epoch {epoch+1}")

            error_count = 0
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                try:
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)

                    if fix_points is None or break_points is None:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation data processing produced None, skipping batch.")
                        error_count += 1
                        continue
                    if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation fix_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation break_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)

                    z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                    z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    alignment_consistency = compute_alignment_consistency_loss(
                        z_0, z_0_aligned, fix_points, self.diffusion_model.autoencoder
                    ) * 0.1
                    total_loss = compute_weighted_diffusion_loss(
                        losses, 0, 1
                    ) + alignment_consistency

                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation loss not finite, skipping batch.")
                        error_count += 1
                        continue

                    batch_metrics = {
                        'total_loss': total_loss.item(),
                        'noise_loss': losses.get('noise_mse', total_loss).item(),
                        'alignment_consistency': alignment_consistency.item(),
                        **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
                    }
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)

                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Val_Loss': f"{total_loss.item():.4f}",
                            'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                        })

                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Validation error: {e}")
                    error_count += 1
                    if self.device == "cuda":
                        torch.cuda.empty_cache()
                    continue

        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print(f"[Epoch {epoch}] WARNING: No valid batches in validation.")
            return {
                'total_loss': float('nan'),
                'noise_loss': float('nan'),
                'alignment_consistency': float('nan')
            }

        return {key: np.mean(values) for key, values in epoch_metrics.items()}

    
    def _train_integrated_stage3(self, num_epochs):
        """Stage 3: Integrated pure diffusion training"""
        
        if not self.check_gpu_health():
            print("GPU status abnormal, cannot start training")
            return

        self._freeze_params_for_stage3()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            self.diffusion_model.train()
            train_metrics = self._train_stage3_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            self.diffusion_model.eval()
            val_metrics = self._validate_stage3_epoch(epoch)
            
            self._log_stage_metrics('stage3', epoch, train_metrics, val_metrics)
            
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage3', epoch, current_loss, is_final=False)
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage3', epoch, current_loss, is_final=True)

            if self.early_stoppers['stage3'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"Stage 3 early stopping at Epoch {epoch+1}")
                break
            
            if self.device == "cuda":
                torch.cuda.empty_cache()


    def _train_stage3_epoch(self, epoch, total_epochs, global_step):
        """Stage 3 pure diffusion training epoch"""
        
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-3: Epoch {epoch+1}/{total_epochs}")
        
        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            if consecutive_errors > 3:
                print(f"[Epoch {epoch} Batch {batch_idx}] Consecutive error limit reached. Stopping epoch early.")
                break
            if critical_error_count > 1:
                print(f"[Epoch {epoch} Batch {batch_idx}] Critical error limit reached, interrupting epoch.")
                break

            try:
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                
                if fix_points is None or break_points is None:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue

                self.optimizers['stage3'].zero_grad()

                with torch.no_grad():
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                        
                        if torch.isnan(z_0).any() or torch.isnan(z_0_aligned).any():
                            print(f"[Epoch {epoch} Batch {batch_idx}] Encoded latents contain NaN, skipping batch.")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Error during encoding: {e}")
                        consecutive_errors += 1
                        continue

                try:
                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    
                    total_loss = compute_weighted_diffusion_loss(
                        losses, global_step + step, len(self.train_dataloader) * total_epochs
                    )
                    
                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is not finite, skipping batch.")
                        consecutive_errors += 1
                        continue
                        
                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Error during forward/loss process: {e}")
                    consecutive_errors += 1
                    continue

                try:
                    total_loss.backward()
                    
                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        self.diffusion_model.parameters(), max_grad_norm
                    )
                    
                    grad_valid = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            grad_valid = False
                            break
                    
                    if not grad_valid:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Invalid gradient detected, dropping batch.")
                        self.optimizers['stage3'].zero_grad()
                        consecutive_errors += 1
                        continue
                    
                    self.optimizers['stage3'].step()
                    self.schedulers['stage3'].step()
                    
                    batch_metrics = {
                        'total_loss': self.safe_tensor_to_float(total_loss),
                        'noise_loss': self.safe_tensor_to_float(losses.get('noise_mse', total_loss)),
                        'grad_norm': self.safe_tensor_to_float(grad_norm),
                        **{k: self.safe_tensor_to_float(v) for k, v in losses.items()}
                    }
                    
                    consecutive_errors = 0
                    critical_error_count = 0
                    
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)
                    
                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Loss': f"{total_loss.item():.4f}",
                            'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                            'GradNorm': f"{grad_norm:.2f}"
                        })
                    
                    step += 1
                    
                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"[Epoch {epoch} Batch {batch_idx}] CUDA runtime error: {str(e)[:100]}")
                        consecutive_errors += 1
                        self.optimizers['stage3'].zero_grad()
                        
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        time.sleep(0.5)
                        continue
                    else:
                        raise e
                        
            except Exception as e:
                print(f"[Epoch {epoch} Batch {batch_idx}] Unexpected training error: {e}")
                consecutive_errors += 1
                critical_error_count += 1
                if critical_error_count > 1:
                    print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                    break
                self.optimizers['stage3'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue
        
        if len(epoch_metrics) == 0 or len(epoch_metrics.get('total_loss', [])) == 0:
            print(f"[Epoch {epoch}] WARNING: No successfully trained batches this epoch.")
            return {
                'total_loss': float('nan'),
                'noise_loss': float('nan'), 
                'grad_norm': float('nan')
            }
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    def _validate_stage3_epoch(self, epoch):
        """Stage 3 Pure Diffusion Validation - Enhanced with comprehensive error checking"""
        
        epoch_metrics = defaultdict(list)
        error_count = 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage3-Val: Epoch {epoch+1}")
            
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                
                if error_count > 2:
                    print(f"[Stage3-Val Epoch {epoch}] Too many validation errors, early termination.")
                    break
                
                try:
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)
                    
                    if fix_points is None or break_points is None:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                        error_count += 1
                        continue
                    
                    if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    
                    if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)
                    
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                        
                        if torch.isnan(z_0).any() or torch.isinf(z_0).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_0 contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                        
                        if torch.isnan(z_0_aligned).any() or torch.isinf(z_0_aligned).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_0_aligned contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during encoding: {e}")
                        error_count += 1
                        continue
                    
                    try:
                        B = z_0_aligned.shape[0]
                        t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                        z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                        predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                        
                        if torch.isnan(z_t).any() or torch.isinf(z_t).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_t contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                        
                        if torch.isnan(predicted_noise).any() or torch.isinf(predicted_noise).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] predicted_noise contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during diffusion forward: {e}")
                        error_count += 1
                        continue
                    
                    try:
                        losses = compute_diffusion_losses_enhanced(
                            predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                        )
                        
                        total_loss = compute_weighted_diffusion_loss(losses, 0, 1)
                        
                        if not torch.isfinite(total_loss):
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] total_loss is not finite, skipping batch.")
                            error_count += 1
                            continue
                        
                        for loss_name, loss_value in losses.items():
                            if torch.is_tensor(loss_value) and not torch.isfinite(loss_value):
                                print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] {loss_name} is not finite, skipping batch.")
                                error_count += 1
                                continue
                                
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during loss computation: {e}")
                        error_count += 1
                        continue
                    
                    try:
                        batch_metrics = {
                            'total_loss': self.safe_tensor_to_float(total_loss),
                            'noise_loss': self.safe_tensor_to_float(losses.get('noise_mse', total_loss)),
                            **{k: self.safe_tensor_to_float(v) for k, v in losses.items()}
                        }
                        
                        valid_metrics = True
                        for key, value in batch_metrics.items():
                            if not isinstance(value, (int, float)) or not np.isfinite(value):
                                print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Invalid metric {key}: {value}")
                                valid_metrics = False
                                break
                        
                        if not valid_metrics:
                            error_count += 1
                            continue
                        
                        for key, value in batch_metrics.items():
                            epoch_metrics[key].append(value)
                        
                        if error_count > 0:
                            error_count = max(0, error_count - 1)
                        
                        if batch_idx % 10 == 0:
                            progress_bar.set_postfix({
                                'Val_Loss': f"{total_loss.item():.4f}",
                                'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                            })
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during metrics recording: {e}")
                        error_count += 1
                        continue
                    
                except Exception as e:
                    print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Unexpected validation error: {e}")
                    error_count += 1
                    if self.device == "cuda":
                        try:
                            torch.cuda.empty_cache()
                        except:
                            pass
                    continue
            
            if len(epoch_metrics) == 0 or len(epoch_metrics.get('total_loss', [])) == 0:
                print(f"[Stage3-Val Epoch {epoch}] WARNING: No valid batches in validation.")
                return {
                    'total_loss': float('nan'),
                    'noise_loss': float('nan')
                }
            
            print(f"[Stage3-Val Epoch {epoch}] Validation completed successfully with {len(epoch_metrics.get('total_loss', []))} valid batches.")
            return self.compute_epoch_metrics_safely(epoch_metrics)


    def safe_tensor_to_float(self, value):
        """Safely convert any value to a Python float"""
        if torch.is_tensor(value):
            return float(value.detach().cpu().numpy())
        elif isinstance(value, (int, float)):
            return float(value)
        else:
            return float('nan')

    def compute_epoch_metrics_safely(self, epoch_metrics):
        """Safely compute epoch metrics"""
        if not epoch_metrics:
            return {
                'total_loss': float('nan'),
                'alignment_loss': float('nan'), 
                'structure_loss': float('nan'),
                'grad_norm': float('nan')
            }
        
        result = {}
        for key, values in epoch_metrics.items():
            if not values:
                result[key] = float('nan')
            else:
                clean_values = [self.safe_tensor_to_float(v) for v in values]
                result[key] = float(np.mean(clean_values))
        
        return result
    

    def _log_stage_metrics(self, stage, epoch, train_metrics, val_metrics):
        """Log stage metrics to TensorBoard"""
        
        writer = self.writers[stage]
        
        for key, value in train_metrics.items():
            writer.add_scalar(f'{stage.capitalize()}_Train/{key}', value, epoch)
            self.stage_metrics[stage]['train'][key].append(value)
        
        for key, value in val_metrics.items():
            writer.add_scalar(f'{stage.capitalize()}_Val/{key}', value, epoch)
            self.stage_metrics[stage]['val'][key].append(value)
        
        current_lr = self.optimizers[stage].param_groups[0]['lr']
        writer.add_scalar(f'{stage.capitalize()}_Learning_Rate', current_lr, epoch)
        self.stage_metrics[stage]['lr'].append(current_lr)
        
        if 'grad_norm' in train_metrics:
            self.stage_metrics[stage]['grad_norm'].append(train_metrics['grad_norm'])
        
        if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
            writer.add_scalars(f'{stage.capitalize()}_Loss_Comparison', {
                'Train': train_metrics['total_loss'],
                'Val': val_metrics['total_loss']
            }, epoch)
        
        print(f"\n{stage.upper()} Epoch {epoch+1} Summary:")
        print(f"Learning Rate: {current_lr:.2e}")
        
        if stage == 'stage1':
            print(f"Train - Alignment Loss: {train_metrics.get('alignment_loss', 0):.4f}, "
                f"Structure Loss: {train_metrics.get('structure_loss', 0):.4f}, "
                f"Grad Norm: {train_metrics.get('grad_norm', 0):.2f}")
            print(f"Val - Alignment Loss: {val_metrics.get('alignment_loss', 0):.4f}")
        else:
            print(f"Train - Total Loss: {train_metrics.get('total_loss', 0):.4f}, "
                f"Noise Loss: {train_metrics.get('noise_loss', 0):.4f}, "
                f"Grad Norm: {train_metrics.get('grad_norm', 0):.2f}")
            print(f"Val - Total Loss: {val_metrics.get('total_loss', 0):.4f}, "
                f"Noise Loss: {val_metrics.get('noise_loss', 0):.4f}")

    def _plot_stage_training_curves(self, stage):
        """Plot training and validation loss curves and save to the corresponding directory."""
        metrics = self.stage_metrics[stage]
        train = metrics['train']
        val = metrics['val']

        if stage == 'stage1':
            loss_key = 'alignment_loss'
            ylabel = 'Alignment Loss'
            title = 'Stage 1 Alignment Loss Curve'
        else:
            loss_key = 'total_loss'
            ylabel = 'Total Loss'
            title = f'Stage {stage[-1]} Diffusion Loss Curve'

        train_loss = train.get(loss_key, [])
        val_loss = val.get(loss_key, [])

        plt.figure(figsize=(8, 6))
        plt.plot(train_loss, label='Train')
        plt.plot(val_loss,   label='Validation')
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)

        save_dir = self.stage_log_dirs[stage]
        save_path = os.path.join(save_dir, f'{stage}_loss_curve.png')
        plt.savefig(save_path)
        plt.close()
        print(f"[{stage}] training loss curve saved: {save_path}")

    def _save_stage_report(self, stage):
        """Save detailed per-epoch training report for each stage.
        Format: logs/stageX_alignment/stageX_report.json
        """
        metrics = self.stage_metrics[stage]
        report = {
            "train": dict(metrics['train']),
            "val": dict(metrics['val']),
            "lr": metrics['lr'],
            "grad_norm": metrics['grad_norm']
        }
        save_dir = self.stage_log_dirs[stage]
        save_path = os.path.join(save_dir, f'{stage}_report.json')

        for key in ["train", "val"]:
            for mname, mvalues in report[key].items():
                report[key][mname] = [float(v) for v in mvalues]

        report["lr"] = [float(v) for v in report["lr"]]
        report["grad_norm"] = [float(v) for v in report["grad_norm"]]

        with open(save_path, 'w') as f:
            json.dump(report, f, indent=2)
        print(f"[{stage}] Detailed training report saved: {save_path}")


    def _generate_multi_stage_report(self):
        """Generate comprehensive multi-stage training report"""
        
        for stage in ['stage1', 'stage2', 'stage3']:
            self._plot_stage_training_curves(stage)
            self._save_stage_report(stage)
        
        comprehensive_report = {
            'multi_stage_training_summary': {
                'total_stages': 3,
                'training_completed': True,
                'stage_summaries': {}
            }
        }
        
        for stage in ['stage1', 'stage2', 'stage3']:
            metrics = self.stage_metrics[stage]
            if metrics['train'].get('total_loss') or metrics['train'].get('alignment_loss'):
                loss_key = 'total_loss' if 'total_loss' in metrics['train'] else 'alignment_loss'
                comprehensive_report['multi_stage_training_summary']['stage_summaries'][stage] = {
                    'epochs_completed': len(metrics['train'][loss_key]),
                    'best_train_loss': min(metrics['train'][loss_key]) if metrics['train'][loss_key] else 0,
                    'best_val_loss': min(metrics['val'][loss_key]) if metrics['val'][loss_key] else 0,
                    'final_lr': metrics['lr'][-1] if metrics['lr'] else 0
                }
        
        with open(f'{self.log_dir}/multi_stage_comprehensive_report.json', 'w') as f:
            json.dump(comprehensive_report, f, indent=2)
        
        print(f"Multi-stage comprehensive training report saved to: {self.log_dir}")


    def ensure_model_on_device(model, device):
        """Ensure the model and all its components are on the specified device"""
        model = model.to(device)
        
        if hasattr(model, 'latent_aligner'):
            model.latent_aligner = model.latent_aligner.to(device)
            print(f"latent_aligner moved to device: {device}")
        
        device_check = {}
        for name, module in model.named_modules():
            if len(list(module.parameters())) > 0:
                param_device = next(module.parameters()).device
                device_check[name] = param_device
        
        return model

多階段訓練策略實作

In [ ]:
def train_integrated_multi_stage_diffusion_model(diffusion_model, train_dataloader, val_dataloader,
                                                stage1_epochs=20, stage2_epochs=80, stage3_epochs=100,
                                                device='cuda', save_path_base="./models/diffusion_multistage",
                                                log_dir="./logs/integrated_multistage_training"):
    """Main function for integrated multi-stage diffusion model training"""
    
    diffusion_model = IntegratedMultiStageTrainingManager.ensure_model_on_device(diffusion_model, device)
    
    diffusion_model = integrate_alignment_to_existing_model(diffusion_model, device)
    
    training_manager = IntegratedMultiStageTrainingManager(
        diffusion_model, train_dataloader, val_dataloader, device, log_dir
    )
    
    stage_metrics = training_manager.run_integrated_multi_stage_training(
        stage1_epochs=stage1_epochs,
        stage2_epochs=stage2_epochs,
        stage3_epochs=stage3_epochs
    )
    
    final_model_path = f"{save_path_base}_integrated_final.pth"
    torch.save({
        'model_state_dict': diffusion_model.state_dict(),
        'stage_metrics': stage_metrics,
        'training_config': {
            'stage1_epochs': stage1_epochs,
            'stage2_epochs': stage2_epochs,
            'stage3_epochs': stage3_epochs
        }
    }, final_model_path)
    
    print(f"Integrated multi-stage training complete, model saved to: {final_model_path}")
    
    return diffusion_model, stage_metrics

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Training Diffusion Model

test

In [ ]:
print("CUDA available:", torch.cuda.is_available())
try:
    test = torch.randn(100, 100).cuda()
    result = test @ test.T
    print("Basic GPU operation successful")
    del test, result
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Basic GPU operation failed: {e}")
    print("Consider restarting Python kernel or system")

In [ ]:
autoencoder = PointCloudAutoencoder(
    num_points=5000, 
    latent_dim=128,
    feature_dim=128
)
autoencoder = autoencoder.to(device)
diffusion_model = EnhancedConditionalDiffusionModel(
        autoencoder=autoencoder,
        feature_dim=128,
        num_points=5000
    )
diffusion_model = integrate_alignment_to_existing_model(diffusion_model, device)
diffusion_model = diffusion_model.to(device)
def minimal_alignment_test():
    device = torch.device('cuda')
    
    test_data = torch.randn(1, 5000, 3).to(device)
    
    try:
        with torch.no_grad():
            z = autoencoder.encode(test_data)
        print(f"AE encoded successfully: {z.shape}")
    except Exception as e:
        print(f"AE encoding failed: {e}")
        return False
    
    # 測試對齊器
    try:
        z_aligned = diffusion_model.latent_aligner.align_latent_space(z)
        print(f"Alignment successful: {z_aligned.shape}")
    except Exception as e:
        print(f"Alignment failed: {e}")
        return False
    
    try:
        B = z_aligned.shape[0]
        t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
        z_t, noise = diffusion_model.add_noise(z_aligned, t)
        
        condition_points = test_data  
        predicted_noise = diffusion_model.forward(z_t, t, condition_points)
        
        print(f"Diffusion model forward successfully: predicted_noise {predicted_noise.shape}, noise {noise.shape}")
    except Exception as e:
        print(f"Diffusion model forward failed: {e}")
        return False
    
    return True

# 執行測試
if minimal_alignment_test():
    print("pass")
else:
    print("component test failed, issue with GPU or model")

In [ ]:
# CUDA_LAUNCH_BLOCKING=1

In [ ]:
torch.cuda.empty_cache()
# train_model(EnhancedConditionalDiffusionModel, dataloader)

In [ ]:
if __name__ == "__main__":
    
    autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=128,
        feature_dim=128
    )


    print("Step 2: Loading pretrained AE weights...")
    pretrained_ae_path = converted_backslash(r"F:\Shawn\Diffusion\MTDC-A Mutilmodal Transformer Diffusion for Cranioplasty\LDM_training_v8_0913_1030\autoencoder_v8_0912_2137\autoencoder_pretrained_v8_0912_2137_best.pth")
    state_dict = torch.load(pretrained_ae_path, map_location="cuda" if torch.cuda.is_available() else 'cpu')
    if 'model_state_dict' in state_dict:
        autoencoder.load_state_dict(state_dict['model_state_dict'])
    else:
        autoencoder.load_state_dict(state_dict)
    autoencoder.eval()
    for param in autoencoder.parameters():
        param.requires_grad = False
    

    diffusion_model = EnhancedConditionalDiffusionModel(
        autoencoder=autoencoder,
        feature_dim=128,
        num_points=5000
    )
    
    models_dir = "./models"
    save_path = os.path.join(models_dir, "LDM_v8_0915_1610.pth")
    log_dir = "./logs/LDM_training_v8_0915_1610"
    
    print("Step 3: Multi-stage Diffusion Model training...")
    trained_model, training_metrics = train_integrated_multi_stage_diffusion_model(
        diffusion_model=diffusion_model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        stage1_epochs=20,
        stage2_epochs=80,
        stage3_epochs=100,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        save_path_base=save_path,
        log_dir=log_dir
    )

    print("Diffusion model training pipeline complete!")
    print(f"Training metrics: {training_metrics}")

繪製圖表：AutoEncoder訓練損失

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

train_losses, train_dcd_losses, train_scale_losses, train_normal_losses, val_losses, val_dcd_losses, val_scale_losses, val_normal_losses

axes[0,0].plot(range(1, len(train_dcd_losses)+1), train_dcd_losses, label="Train DCD", color="blue")
axes[0,0].plot(range(1, len(val_dcd_losses)+1), val_dcd_losses, label="Valid DCD", color="orange")
axes[0,0].set_title("DCD Loss")
axes[0,0].set_yscale("log")
axes[0,0].grid(True)
axes[0,0].legend()

axes[0,1].plot(range(1, len(train_scale_losses)+1), train_scale_losses, label="Train Scale", color="blue")
axes[0,1].plot(range(1, len(val_scale_losses)+1), val_scale_losses, label="Valid Scale", color="orange")
axes[0,1].set_title("Scale Loss")
axes[0,1].set_yscale("log")
axes[0,1].grid(True)
axes[0,1].legend()

axes[0,2].plot(range(1, len(train_normal_losses)+1), train_normal_losses, label="Train Normal", color="blue")
axes[0,2].plot(range(1, len(val_normal_losses)+1), val_normal_losses, label="Valid Normal", color="orange")
axes[0,2].set_title("Normal Loss")
axes[0,2].set_yscale("log")
axes[0,2].grid(True)
axes[0,2].legend()

axes[1,0].plot(range(1, len(train_losses)+1), train_losses, label="Train Total", color="blue")
axes[1,0].plot(range(1, len(val_losses)+1), val_losses, label="Valid Total", color="orange")
axes[1,0].set_title("Total Loss")
axes[1,0].set_yscale("log")
axes[1,0].grid(True)
axes[1,0].legend()

axes[1,1].plot(range(1, len(train_losses)+1), train_losses, label="Train Total", color="blue", alpha=0.7)
axes[1,1].plot(range(1, len(train_dcd_losses)+1), train_dcd_losses, label="Train DCD", color="orange", alpha=0.7)
axes[1,1].plot(range(1, len(train_scale_losses)+1), train_scale_losses, label="Train Scale", color="green", alpha=0.7)
axes[1,1].plot(range(1, len(train_normal_losses)+1), train_normal_losses, label="Train Normal", color="red", alpha=0.7)
axes[1,1].set_title("All Train Losses")
axes[1,1].set_yscale("log")
axes[1,1].grid(True)
axes[1,1].legend()

axes[1,2].plot(range(1, len(val_losses)+1), val_losses, label="Valid Total", color="blue", alpha=0.7)
axes[1,2].plot(range(1, len(val_dcd_losses)+1), val_dcd_losses, label="Valid DCD", color="orange", alpha=0.7)
axes[1,2].plot(range(1, len(val_scale_losses)+1), val_scale_losses, label="Valid Scale", color="green", alpha=0.7)
axes[1,2].plot(range(1, len(val_normal_losses)+1), val_normal_losses, label="Valid Normal", color="red", alpha=0.7)
axes[1,2].set_title("All Valid Losses")
axes[1,2].set_yscale("log")
axes[1,2].grid(True)
axes[1,2].legend()

fig.suptitle("Training for LDM_v7_0823_1613", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
dcd_ae = np.array(dcd_ae)
mse_losses_ae = np.array(mse_losses_ae)
epochs = range(1, len(dcd_ae) + 1)

difference = dcd_ae - mse_losses_ae

fig, ax1 = plt.subplots()

ax1.plot(epochs, dcd_ae, label="DCD Loss", color="yellow", alpha=1)
ax1.plot(epochs, mse_losses_ae, label="CD Loss", color="blue", alpha=0.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (Log Scale)", color="blue")
ax1.set_yscale('log')
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(epochs, difference, label="DCD - CD Difference", color="red", linestyle="--")
ax2.set_ylabel("Difference", color="red")
ax2.legend(loc="upper right")

plt.title("DCD Loss vs CD Loss with Difference")
plt.show()

relative_difference = (difference / cd_losses_ae) * 100
plt.figure()
plt.plot(epochs, relative_difference, label="Relative Difference (%)", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Relative Difference (%)")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.title("Relative Difference between DCD Loss and CD Loss")
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(range(1, len(dcd_ae)+1), dcd_ae, label="Total Loss", color="blue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Total Loss", color = "tab:blue")
ax1.tick_params(axis="y", labelcolor = "tab:blue")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(range(1, len(mse_losses_ae) + 1), mse_losses_ae, label="MSE Loss", color="tab:orange")
ax2.plot(range(1, len(cd_losses_ae) + 1), cd_losses_ae, label="CD Loss", color="tab:green")
ax2.set_ylabel("MSE & CD Loss", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.show()

In [ ]:
num_epochs = 200
lr = 5e-5
save_path="ddpm_v4_04_06.pth"
losses = []
accuracies = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnhancedConditionalDiffusionModel()
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for break_points, fix_points, norm_params in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_size = break_points.size(0)
        t = torch.randint(0, model.num_timesteps, (batch_size,), device=device)
        
        break_points = break_points + torch.randn_like(break_points) * 0.01
        fix_points = fix_points + torch.randn_like(fix_points) * 0.01
        
        x_t, noise = model.add_noise(fix_points, t)
        pred_noise = model(x_t, t, break_points)
        
        noise_loss = F.mse_loss(pred_noise, noise)
        pred_points = model.sample(break_points, denormalize_params=norm_params)
        loss = noise_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(dataloader):.4f}")
    loss_per_epoch = total_loss / len(dataloader)
    losses.append(loss_per_epoch)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_per_epoch:.4f}")
    
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"{save_path.split('.')[0]}_epoch{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': loss_per_epoch
        }, checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

Continue training from Epoch 50

In [ ]:
num_epochs = 100
continue_epochs = 50
lr = 5e-5
save_path="ddpm_v4_04_06.pth"
losses = []
accuracies = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weight_path = converted_backslash(r"C:\Users\user\Desktop\小夜\GAN\cranioplasty_gan_2\LDM_training_v1_0428_1519\LDM_v1_0430_2021_epoch200.pth")
state_dict = torch.load(weight_path, map_location=device)
model = EnhancedConditionalDiffusionModel()
model.to(device)
model.load_state_dict(state_dict)

optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
checkpoint = torch.load(weight_path, map_location=device)

In [ ]:
for epoch in range(continue_epochs, num_epochs):
    model.train()
    total_loss = 0
    for break_points, fix_points, norm_params in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_size = break_points.size(0)
        t = torch.randint(0, model.num_timesteps, (batch_size,), device=device)
        
        break_points = break_points + torch.randn_like(break_points) * 0.01 
        fix_points = fix_points + torch.randn_like(fix_points) * 0.01
        
        x_t, noise = model.add_noise(fix_points, t)
        pred_noise = model(x_t, t, break_points)
        
        noise_loss = F.mse_loss(pred_noise, noise)
        pred_points = model.sample(break_points, denormalize_params=norm_params)
        cd_loss = chamfer_distance(pred_points, fix_points)
        loss = noise_loss + 0.1 * cd_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    loss_per_epoch = total_loss / len(dataloader)
    losses.append(loss_per_epoch)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_per_epoch:.4f}")
    
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"{save_path.split('.')[0]}_epoch{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': loss_per_epoch
        }, checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Evaluation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
break_points, fix_points, norm_params = dataset[00]

In [ ]:
autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=256,
        feature_dim=128
)

diffusion_model = EnhancedConditionalDiffusionModel(
    autoencoder=autoencoder,
    feature_dim=256,
    num_points=5000
)

weight_path = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\models\LDM_v8_0904_1107_best.pth")
checkpoint = torch.load(weight_path, map_location=device)
diffusion_model.to(device)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    diffusion_model.load_state_dict(checkpoint['model_state_dict'])
else:
    diffusion_model.load_state_dict(checkpoint)

diffusion_model.eval()

create a method to show plot lines for loss ( Run Graph of Loss )

In [ ]:
# def plot_lines(losses, start_epoch=1, step=10):
#   epochs = list(range(start_epoch, start_epoch + len(losses)*step, step))
#   arr = np.array(losses)

#   fig, ax = plt.subplots() # 生成圖表和對應的座標軸
#   ax.set_xlabel('Epoch') # 設x軸為Epoch(訓練週期)
#   ax.set_ylabel('Loss') # 設y軸為Loss(損失值)
#   ax.plot(epochs, arr, marker='', linestyle='-', label="loss")
#   ax.legend()
#   ax.grid(True)
#   # return fig

In [ ]:
def plot_lines_per_10_epoch(losses1, step=10, title="title"):
    losses1 = np.array(losses1)
    epochs = np.arange(1, len(losses1) + 1)
    
    max_epoch = len(losses1)
    
    x_ticks = np.arange(0, max_epoch + 1, step)
    
    selected_epochs = np.arange(10, max_epoch + 1, step)
    if selected_epochs[0] != 1:
        selected_epochs = np.insert(selected_epochs, 0, 1)
        
    selected_losses = losses1[selected_epochs - 1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    
    ax.plot(epochs, losses1, linestyle="-", alpha=0.4, label="All Losses")
    
    ax.plot(selected_epochs, selected_losses, marker='o', color="orange", 
            linestyle="-", label="DCD Loss")
    
    ax.legend()
    ax.grid(True)
    
    plt.xticks(x_ticks)
    
    plt.xlim(0, max_epoch)
    
    title = f"Loss per 10 Epochs for Diffusion Model"
    plt.title(title)
    plt.tight_layout()
    return fig, selected_epochs

In [ ]:
def visualize_point_cloud_pair(brk=None, org=None, gen=None, title="Point Cloud Comparison"):
    """Visualize broken skull and generated implant (and original if provided)"""
    pcds = []
    
    if brk is not None:
        break_pcd = o3d.io.read_point_cloud(brk)
        break_pcd.paint_uniform_color([0.7, 0.7, 0.7])
        pcds.append(break_pcd)
    
    if org is not None:
        org_pcd = o3d.io.read_point_cloud(org)
        org_pcd.paint_uniform_color([0.8, 0, 0])
        pcds.append(org_pcd)

    if gen is not None:
        gen_pcd = o3d.geometry.PointCloud()
        gen_pcd.points = o3d.utility.Vector3dVector(gen)
        gen_pcd.paint_uniform_color([0, 0.8, 0])
        pcds.append(gen_pcd)
    
    o3d.visualization.draw_geometries(pcds, window_name=title)

In [ ]:
def save_merged_point_cloud(output_path, brk=None, org=None, gen=None, title="Point Cloud Comparison"):
    """Visualize broken skull and generated implant (and original if provided)"""
    pcds = []
    
    if brk is not None:
        break_pcd = o3d.io.read_point_cloud(brk)
        break_pcd.paint_uniform_color([0.7, 0.7, 0.7])
        pcds.append(break_pcd)
    
    if org is not None:
        org_pcd = o3d.io.read_point_cloud(org)
        org_pcd.paint_uniform_color([0.8, 0, 0])
        pcds.append(org_pcd)

    if gen is not None:
        gen_pcd = o3d.geometry.PointCloud()
        gen_pcd.points = o3d.utility.Vector3dVector(gen)
        gen_pcd.paint_uniform_color([0, 0.8, 0])
        pcds.append(gen_pcd)
    
    if pcds is None:
        print("No point clouds available, merge error.")
        return
    
    pcds_combined = pcds[0]
    for pcd in pcds[1:]:
        pcds_combined += pcd
    
    o3d.io.write_point_cloud(output_path, pcds_combined)
    print("Merged point cloud has been successfully saved to the specified path.")

In [ ]:
def visualize_xyz_concatenate_2D(break_file, fix_points, view_plane="xy"):
    break_points = np.loadtxt(break_file, delimiter=" ")
    
    if view_plane == "xy":
        break_x, break_y = break_points[:, 0], break_points[:, 1]
        fix_x, fix_y = fix_points[:, 0], fix_points[:, 1]
        xlabel, ylabel = "X Axis", "Y Axis"
    elif view_plane == "xz":
        break_x, break_y = break_points[:, 0], break_points[:, 2]
        fix_x, fix_y = fix_points[:, 0], fix_points[:, 2]
        xlabel, ylabel = "X Axis", "Z Axis"
    else:
        raise ValueError("'view_plane must be xy or xz'")
    
    plt.figure(figsize=(10, 8))
    plt.scatter(break_x, break_y, s=1, c="gray", label="Break Skull")
    plt.scatter(fix_x, fix_y, s=1, c="blue", label="Fix Skull")
    
    plt.xlabel('X Axis')
    plt.ylabel('Y Axis')
    plt.title('Break & Fix Skull')
    plt.legend()
    plt.grid(False)

    plt.show()

可視化自編碼器訓練結果

In [ ]:
import open3d as o3d
def visualize_point_cloud(pred_points, fix_points=None, break_points=None):
    pred_pcd = o3d.geometry.PointCloud()
    pred_pcd.points = o3d.utility.Vector3dVector(pred_points.cpu().numpy()[0])
    pred_pcd.paint_uniform_color([0, 0.8, 0.5])
    
    gt_pcd = o3d.geometry.PointCloud()
    gt_pcd.points = o3d.utility.Vector3dVector(fix_points.cpu().numpy()[0])
    gt_pcd.paint_uniform_color([1, 0, 0])
    
    # brk_pcd = o3d.geometry.PointCloud()
    # brk_pcd.points = o3d.utility.Vector3dVector(break_points.cpu().numpy()[0])
    # brk_pcd.paint_uniform_color([0.7, 0.7, 0.7])

    o3d.visualization.draw_geometries([pred_pcd, gt_pcd])
    # o3d.visualization.draw_geometries([pred_pcd, gt_pcd, brk_pcd])
    # o3d.visualization.draw_geometries([gt_pcd])
    # o3d.visualization.draw_geometries([pred_pcd])

In [ ]:
autoencoder = PointCloudAutoencoder(num_points=5000, latent_dim=128, feature_dim=128)
autoencoder = autoencoder.to(device)
weight_path = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth")
checkpoint = torch.load(weight_path, map_location=device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    autoencoder.load_state_dict(checkpoint['model_state_dict'])
else:
    autoencoder.load_state_dict(checkpoint)

In [ ]:
autoencoder.eval()
with torch.no_grad():
    for _, fix_points, _ in train_dataloader:
        fix_points = fix_points.to(device)
        recon, _ = autoencoder(fix_points)
        visualize_point_cloud(recon, fix_points)
        break

In [ ]:
with torch.no_grad():
        break_points_batch = break_points.to(device)
        if break_points.dim() == 2:
            break_points_batch = break_points.unsqueeze(0).to(device)
        else:
            break_points_batch = break_points.to(device)
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        print("break_points_batch shape:", break_points_batch.shape)

        generated_points = diffusion_model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        if generated_points.dim() == 3:
            if generated_points.shape[0] == 1:
                generated_points = generated_points.squeeze(0)

generated_points_00 = generated_points.cpu().numpy()

In [ ]:
generated_points_00 = generated_points

In [ ]:
generated_points_23 = generated_points

In [ ]:
gen_points_dict = {}
n = 99

for i in range(n+1):
    break_points, fix_points, norm_params = dataset[i]
    with torch.no_grad():
        break_points_batch = break_points.unsqueeze(0).to(device)
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        if generated_points.dim() == 3:
            if generated_points.shape[0] == 1:
                generated_points = generated_points.squeeze(0)
            elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:
                generated_points = generated_points.squeeze(0)
            elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:
                generated_points = generated_points[0]

    gen_points_dict[i] = generated_points.squeeze(0).cpu().numpy()

In [ ]:
print(break_points.shape)
print(fix_points.shape)
print(generated_points.shape)

visualization

In [ ]:
print("break_points shape:", break_points.shape, "dtype:", break_points.dtype)
print("generated_points shape:", generated_points.shape, "dtype:", generated_points.dtype)
print("fix_points shape:", fix_points.shape, "dtype:", fix_points.dtype)

In [ ]:
file_path_break00_10000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_10000\000_break.xyz")
file_path_break00_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_70
file_path_fix00_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\000_fix.xyz")

In [ ]:
# 視覺化v1
visualize_point_cloud_pair(
    brk = file_path_break00_10000,
    org = file_path_fix00_5000,
    # gen = gen_points_dict[8],
    gen = generated_points_00,
    title="Skull Repair Visualization (Epoch 200)"
)

In [ ]:
output_dir = converted_backslash(r"C:\dataset\Skull Fix & Break\training_set\generated_output_xyz")
output_filename = f"ddpm_v3_0324_1421_epoch140_skull47_5000points.xyz"
save_path = os.path.join(output_dir, output_filename)

In [ ]:
save_merged_point_cloud(
    output_path = save_path,
    brk = file_path_break00_30000,
    gen = gen_points_dict[47],
)

## Visualizing the Model

In [ ]:
import torch
from torch.autograd import Variable
from torchviz import make_dot

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EnhancedConditionalDiffusionModel(
    feature_dim=256,
    beta_schedule='cosine',
    num_points=5000,
    latent_dim=16,
    latent_feature_dim=64
).to(device)

model.train()

batch_size = 2
latent_dim = 16
latent_feature_dim = 64
num_points = 5000

z_t = torch.randn(batch_size, latent_dim, latent_feature_dim).to(device)
t = torch.randint(0, 500, (batch_size,), device=device).float() / 500
condition_points = torch.randn(batch_size, num_points, 3).to(device)

z_t = Variable(z_t, requires_grad=True)
t = Variable(t, requires_grad=True)
condition_points = Variable(condition_points, requires_grad=True)

output = model(z_t, t, condition_points)

dot = make_dot(output, params=dict(model.named_parameters()))

dot.format = 'png'
dot.render("enhanced_diffusion_model_architecture", view=True)

print("Model architecture visualization has been generated as 'enhanced_diffusion_model_architecture.png'.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EnhancedConditionalDiffusionModel(
    feature_dim=256,
    beta_schedule='cosine',
    num_points=5000,
    latent_dim=16,
    latent_feature_dim=64
).to(device)

model.train()

batch_size = 1
latent_dim = 16
latent_feature_dim = 64
num_points = 500

z_t = torch.randn(batch_size, latent_dim, latent_feature_dim).to(device)
t = torch.tensor([250], device=device).float() / 500
condition_points = torch.randn(batch_size, num_points, 3).to(device)

z_t = Variable(z_t, requires_grad=True)
t = Variable(t, requires_grad=True)
condition_points = Variable(condition_points, requires_grad=True)

output = model(z_t, t, condition_points)

dot = make_dot(output, params={name: param for name, param in model.named_parameters() if 'weight' in name})

dot.attr(rankdir='TB')
dot.attr('node', shape='box', style='filled', fillcolor='lightblue')
dot.attr('edge', color='blue')

dot.format = 'png'
dot.render("simplified_enhanced_diffusion_model_architecture", view=True)

print("Simplified model architecture visualization has been generated as 'simplified_enhanced_diffusion_model_architecture.png'.")

## Test

In [ ]:
test_path = r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz"
xyz_testing_data = converted_backslash(test_path)
print(xyz_testing_data)

In [ ]:
if not os.path.exists(xyz_testing_data):
    raise FileNotFoundError(f"Directory not found: {xyz_testing_data}, please verify the path.")
print(os.listdir(xyz_testing_data))

In [ ]:
dataset_test = SkullDataset2(data_dir=xyz_testing_data, num_points=5000)
if len(dataset_test) == 0:
    raise ValueError("Error: SkullDataset2 loaded 0 samples, please verify the data source.")
dataloader = DataLoader(
    dataset_test, 
    batch_size=8,
    shuffle=True, 
    collate_fn=collate_fn,
)
print(len(dataloader))

Single Data Testing

In [ ]:
break_points, fix_points, norm_params = dataset_test[00]

Test AutoEncoder

In [ ]:
# AE test
model.autoencoder.eval()
with torch.no_grad():
    for _, fix_points, _ in dataloader:
        fix_points = fix_points.to(device)
        recon, _ = model.autoencoder(fix_points)
        visualize_point_cloud(recon, fix_points)
        break

Test Diffusion Model

In [ ]:
with torch.no_grad():
    break_points_batch = break_points.unsqueeze(0).to(device)
    norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
    generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
    print("Raw generated_points shape:", generated_points.shape)
    
    if generated_points.dim() == 3:
        if generated_points.shape[0] == 1:
            generated_points = generated_points.squeeze(0)
        elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:
            generated_points = generated_points.squeeze(0)
        elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:
            generated_points = generated_points[0]

gereated_points_test_00 = generated_points.squeeze(0).cpu().numpy()

Whole dataset Testing

In [ ]:
gen_points_test_dict = {}
n = 99

for i in range(n+1):
    break_points, fix_points, norm_params = dataset_test[i]
    with torch.no_grad():
        break_points_batch = break_points.unsqueeze(0).to(device)
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        if generated_points.dim() == 3:
            if generated_points.shape[0] == 1:
                generated_points = generated_points.squeeze(0)
            elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:
                generated_points = generated_points.squeeze(0)
            elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:
                generated_points = generated_points[0]

    gen_points_test_dict[i] = generated_points.squeeze(0).cpu().numpy()

In [ ]:
print(gen_points_test_dict)

In [ ]:
generated_points_test_99 = generated_points

In [ ]:
generated_points_test_00 = generated_points

In [ ]:
test_00_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\test_set_for_participants\output_xyz\000.xyz")
test_23_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\023.xyz")
test_50_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\050.xyz")
test_99_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\099.xyz")

In [ ]:
visualize_point_cloud_pair(
    brk = test_00_5000,
    # org = file_path_fix00_5000,
    # gen = gen_points_test_dict[00],
    gen = gereated_points_test_00,
    title="Skull Repair Visualization (Epoch 100)"
)

In [ ]:
def evaluate_model(model, dataset, num_samples=5):
    """Evaluate model on test samples and visualize results"""
    model.eval()
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            break_points, fix_points, norm_params = dataset[i]
            
            break_points = break_points.unsqueeze(0).to(device)
            fix_points = fix_points.unsqueeze(0).to(device)
            norm_params = {k: v.unsqueeze(0).to(device) for k,v in norm_params.keys()}
            
            generated_points = model.sample(break_points, num_points=5000, denormalize_params=norm_params)
            generated_points = generated_points.squeeze(0).cpu().numpy()
            
            break_points_np = break_points.squeeze(0).cpu().numpy()
            fix_points_np = fix_points.squeeze(0).cpu().numpy()
            
            visualize_point_cloud_pair(
                brk=break_points_np,
                org=fix_points_np,
                gen=generated_points,
                title=f"Sample {i+1}: Skull Repair Comparison"
            )

model = EnhancedConditionalDiffusionModel(feature_dim=256, beta_schedule='linear')
model.load_state_dict(state_dict)
model.to(device)
dataset = SkullDataset2(data_dir=xyz_training_data, num_points=5000)
evaluate_model(model, dataset, num_samples=3)